[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/information_theory/04_kl_divergence_and_f_divergences/exercises.ipynb)

# Module 04 — Exercises: KL Divergence and f-Divergences

Thirty-one solved problems in four tiers. Every problem carries a statement, a one-line intuition,
a stepwise solution, a boxed answer, a key takeaway, and — wherever the answer is numeric or
algorithmic — a code cell that recomputes it and prints the check.

Theorem, proof and example numbers refer to
[first_principles.ipynb](first_principles.ipynb). Symbols follow
[the notation register](../../docs/notation.md): $D_{\mathrm{KL}}(P \parallel Q)$ with `\parallel`,
$H_{\times}(p, q)$ for cross-entropy, $H(X, Y)$ reserved for joint entropy.

Divergences are quoted in nats unless a problem says bits.

The preamble below is shared by every code cell in this notebook.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

from scipy import integrate, special, stats

EPS = np.finfo(float).eps
BITS = 1.0 / np.log(2.0)


def kl(p, q):
    """D_KL(P || Q) in nats, with 0 ln(0/q) = 0 and p ln(p/0) = +inf."""
    p = np.asarray(p, float)
    q = np.asarray(q, float)
    live = p > 0.0
    if np.any(q[live] == 0.0):
        return np.inf
    return float(np.sum(p[live] * np.log(p[live] / q[live])))


def tv(p, q):
    return 0.5 * float(np.abs(np.asarray(p, float) - np.asarray(q, float)).sum())


def chi2(p, q):
    p = np.asarray(p, float)
    q = np.asarray(q, float)
    return float(np.sum((p - q) ** 2 / q))


def hellinger2(p, q):
    p = np.asarray(p, float)
    q = np.asarray(q, float)
    return float(np.sum((np.sqrt(p) - np.sqrt(q)) ** 2))


def js(p, q):
    m = 0.5 * (np.asarray(p, float) + np.asarray(q, float))
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


def gauss_kl(m1, s1, m2, s2):
    return np.log(s2 / s1) + (s1 ** 2 + (m1 - m2) ** 2) / (2 * s2 ** 2) - 0.5


print(f"machine epsilon = {EPS:.4e}")

machine epsilon = 2.2204e-16


## L0 — Concept Checks

### Problem L0.1 — Both directions of a Bernoulli divergence

**Statement.** Let $P = \mathrm{Ber}(0.9)$ and $Q = \mathrm{Ber}(0.5)$. Compute
$D_{\mathrm{KL}}(P \parallel Q)$ and $D_{\mathrm{KL}}(Q \parallel P)$ in bits.

**Intuition.** The first argument supplies the frequencies and the second supplies the code, so
swapping them asks a different question.

**Solution.**

*Step 1.* Against the uniform $Q$ every $-\log_2 q(x)$ equals $1$, so
$D_{\mathrm{KL}}(P \parallel Q) = 1 - H_b(0.9)$ bits, with $H_b(0.9) = 0.468996$ bits.

*Step 2.* Hence $D_{\mathrm{KL}}(P \parallel Q) = 0.531004$ bits.

*Step 3.* In the other direction the reference is skewed:
$D_{\mathrm{KL}}(Q \parallel P) = \tfrac12 \log_2\tfrac{0.5}{0.9} + \tfrac12 \log_2\tfrac{0.5}{0.1}
= \tfrac{1}{2}\log_2 \tfrac{25}{9} = 0.736966$ bits.

$$
\boxed{D_{\mathrm{KL}}(P \parallel Q) = 0.531004 \text{ bits}, \qquad D_{\mathrm{KL}}(Q \parallel P) = 0.736966 \text{ bits}}
$$

**Key takeaway.** Describing a near-deterministic source with a uniform code wastes little;
describing a fair coin with a code that reserves $\log_2 10 \approx 3.32$ bits for one face wastes
a great deal.

In [2]:
P = np.array([0.9, 0.1])
Q = np.array([0.5, 0.5])
print(f"D(P||Q) = {kl(P, Q) * BITS:.6f} bits   D(Q||P) = {kl(Q, P) * BITS:.6f} bits")
print(f"1 - H_b(0.9) = {1 - stats.entropy(P, base=2):.6f} bits")
print(f"0.5*log2(25/9) = {0.5 * np.log2(25 / 9):.6f} bits")
assert abs(kl(P, Q) * BITS - 0.531004) < 5e-7
assert abs(kl(Q, P) * BITS - 0.736966) < 5e-7

D(P||Q) = 0.531004 bits   D(Q||P) = 0.736966 bits
1 - H_b(0.9) = 0.531004 bits
0.5*log2(25/9) = 0.736966 bits


### Problem L0.2 — When the divergence is infinite

**Statement.** Let $P = (0.4, 0.4, 0.2)$ and $Q = (0.5, 0.5, 0)$ on $\{a, b, c\}$. Compute both
directions in bits.

**Intuition.** Absolute continuity (Definition 3.1) is one-directional, so one of the two answers
can be infinite while the other is small.

**Solution.**

*Step 1.* $p(c) = 0.2 \gt 0$ while $q(c) = 0$, so $P \not\ll Q$ and by Definition 3.2 the forward
divergence is $+\infty$.

*Step 2.* In reverse, $Q \ll P$: the outcome $c$ simply carries $q(c) = 0$, and $0\log 0 = 0$.

*Step 3.* $D_{\mathrm{KL}}(Q \parallel P) = 2 \times 0.5\log_2\frac{0.5}{0.4} = \log_2 1.25 = 0.321928$ bits.

$$
\boxed{D_{\mathrm{KL}}(P \parallel Q) = +\infty, \qquad D_{\mathrm{KL}}(Q \parallel P) = 0.321928 \text{ bits}}
$$

**Key takeaway.** A model that omits a real outcome is infinitely bad in the forward direction and
perfectly finite in the reverse one — the reason maximum likelihood never assigns zero probability
to observed data.

In [3]:
P2 = np.array([0.4, 0.4, 0.2])
Q2 = np.array([0.5, 0.5, 0.0])
print(f"D(P||Q) = {kl(P2, Q2)}")
print(f"D(Q||P) = {kl(Q2, P2) * BITS:.6f} bits   log2(1.25) = {np.log2(1.25):.6f}")
assert np.isinf(kl(P2, Q2))
assert abs(kl(Q2, P2) * BITS - np.log2(1.25)) < 1e-12

D(P||Q) = inf
D(Q||P) = 0.321928 bits   log2(1.25) = 0.321928


### Problem L0.3 — Generators are defined only up to an affine term

**Statement.** Verify that $f(t) = t \ln t$ generates $D_{\mathrm{KL}}$, satisfies $f(1) = 0$ and is
strictly convex, and show that $f_c(t) = t\ln t + c\,(t-1)$ generates the same divergence for every
$c \in \mathbb{R}$.

**Intuition.** The affine term integrates to zero because both distributions are normalized.

**Solution.**

*Step 1.* Substituting $t = p(x)/q(x)$ into Definition 3.3,
$\sum_x q(x)\frac{p(x)}{q(x)}\ln\frac{p(x)}{q(x)} = \sum_x p(x)\ln\frac{p(x)}{q(x)} = D_{\mathrm{KL}}(P \parallel Q)$.

*Step 2.* $f(1) = 0$, $f'(t) = \ln t + 1$ and $f''(t) = 1/t \gt 0$ on $(0,\infty)$: strictly convex.

*Step 3.* The extra term contributes
$\sum_x q(x)\,c\left(\frac{p(x)}{q(x)} - 1\right) = c(1 - 1) = 0$, and $f_c(1) = 0$ with
$f_c'' = f''$.

$$
\boxed{f(t) = t\ln t \Rightarrow D_f = D_{\mathrm{KL}}; \qquad f(t) + c(t-1) \text{ gives the same divergence}}
$$

**Key takeaway.** This is the $\kappa = 1$ case of Proposition 4.10. Choosing $c = -f'(1) = -1$
makes the generator nonnegative and produces $t - 1 - \ln t$ reversed, the estimator studied in
Problem L2.2.

In [4]:
Pa = np.array([0.5, 0.3, 0.2])
Qa = np.array([0.1, 0.2, 0.7])
t_ratio = Pa / Qa
base = float(np.sum(Qa * (t_ratio * np.log(t_ratio))))
print(f"D_f with f(t) = t ln t      : {base:.12f}")
print(f"D_KL directly               : {kl(Pa, Qa):.12f}")
for c in (-3.0, -1.0, 0.0, 2.5):
    shifted = float(np.sum(Qa * (t_ratio * np.log(t_ratio) + c * (t_ratio - 1.0))))
    print(f"  c = {c:5.1f} -> D_f = {shifted:.12f}   difference = {abs(shifted - base):.2e}")
    assert abs(shifted - base) < 1e-12

D_f with f(t) = t ln t      : 0.675805894950
D_KL directly               : 0.675805894950
  c =  -3.0 -> D_f = 0.675805894950   difference = 4.44e-16
  c =  -1.0 -> D_f = 0.675805894950   difference = 2.22e-16
  c =   0.0 -> D_f = 0.675805894950   difference = 0.00e+00
  c =   2.5 -> D_f = 0.675805894950   difference = 2.22e-16


### Problem L0.4 — Total variation and a Pinsker check

**Statement.** For $P = \mathrm{Ber}(0.9)$ and $Q = \mathrm{Ber}(0.5)$ compute
$\mathrm{TV}(P, Q)$ and verify Theorem 4.7, $\mathrm{TV} \le \sqrt{D_{\mathrm{KL}}/2}$, with
$D_{\mathrm{KL}}$ in nats.

**Intuition.** Pinsker converts a divergence, which is easy to compute, into a statement about
probabilities of events, which is what one usually wants.

**Solution.**

*Step 1.* $\mathrm{TV}(P,Q) = \tfrac12\left(\lvert 0.9 - 0.5\rvert + \lvert 0.1 - 0.5 \rvert\right) = 0.4$.

*Step 2.* From Problem L0.1, $D_{\mathrm{KL}}(P \parallel Q) = 0.531004$ bits $= 0.368064$ nats.

*Step 3.* $\sqrt{0.368064/2} = \sqrt{0.184032} = 0.428990 \ge 0.4$.

$$
\boxed{\mathrm{TV} = 0.400000 \; \le \; 0.428990 = \sqrt{D_{\mathrm{KL}}/2}}
$$

**Key takeaway.** The slack is about $7$ percent here; by Section 7.5 of the theory notebook the
bound is attained only in the limit $P \to Q$ along the symmetric Bernoulli path.

In [5]:
bound = np.sqrt(0.5 * kl(P, Q))
print(f"TV = {tv(P, Q):.6f}   sqrt(KL/2) = {bound:.6f}   slack = {bound - tv(P, Q):.6f}"
      f"   relative slack = {(bound - tv(P, Q)) / tv(P, Q) * 100:.1f} percent")
assert tv(P, Q) <= bound

TV = 0.400000   sqrt(KL/2) = 0.428990   slack = 0.028990   relative slack = 7.2 percent


### Problem L0.5 — Relative entropy is not a metric

**Statement.** Exhibit a pair violating symmetry and a triple violating the triangle inequality for
$D_{\mathrm{KL}}$.

**Intuition.** A divergence is a directed cost, not a distance; nothing in Definition 3.2 makes the
two arguments interchangeable.

**Solution.**

*Step 1 (symmetry).* Problem L0.1 already gives $0.368064 \neq 0.510826$ nats for
$\mathrm{Ber}(0.9)$ against $\mathrm{Ber}(0.5)$.

*Step 2 (triangle).* Take $P = \mathrm{Ber}(0.1)$, $R = \mathrm{Ber}(0.5)$, $S = \mathrm{Ber}(0.9)$.
Then $D_{\mathrm{KL}}(P \parallel S) = 0.1\ln\tfrac{1}{9} + 0.9\ln 9 = 0.8 \ln 9 = 1.757780$, while
$D_{\mathrm{KL}}(P \parallel R) + D_{\mathrm{KL}}(R \parallel S) = 0.368064 + 0.510826 = 0.878890$.

*Step 3.* $1.757780 \gt 0.878890$: the detour through the midpoint is cheaper than the direct
route by exactly a factor of two.

$$
\boxed{D_{\mathrm{KL}}(P \parallel S) = 1.757780 \; \gt \; 0.878890 = D_{\mathrm{KL}}(P \parallel R) + D_{\mathrm{KL}}(R \parallel S)}
$$

**Key takeaway.** Only the local quadratic form of $D_{\mathrm{KL}}$ is a metric structure — the
Fisher information of Section 7.3 — and only to second order.

In [6]:
Pb, Rb, Sb = np.array([0.1, 0.9]), np.array([0.5, 0.5]), np.array([0.9, 0.1])
direct = kl(Pb, Sb)
detour = kl(Pb, Rb) + kl(Rb, Sb)
print(f"asymmetry : D(Ber(0.9)||Ber(0.5)) = {kl(Sb, Rb):.6f}   "
      f"D(Ber(0.5)||Ber(0.9)) = {kl(Rb, Sb):.6f}")
print(f"triangle  : D(P||S) = {direct:.6f}   D(P||R) + D(R||S) = {detour:.6f}"
      f"   ratio = {direct / detour:.4f}")
assert kl(Sb, Rb) != kl(Rb, Sb)
assert direct > detour

asymmetry : D(Ber(0.9)||Ber(0.5)) = 0.368064   D(Ber(0.5)||Ber(0.9)) = 0.510826
triangle  : D(P||S) = 1.757780   D(P||R) + D(R||S) = 0.878890   ratio = 2.0000


### Problem L0.6 — Total variation is an $f$-divergence

**Statement.** Show that $f(t) = \tfrac12\lvert t - 1 \rvert$ is a generator and that
$D_f = \mathrm{TV}$.

**Intuition.** The template of Definition 3.3 multiplies $f(p/q)$ by $q$, which clears the
denominator.

**Solution.**

*Step 1.* $f$ is convex (an absolute value of an affine function) and $f(1) = 0$, so it is a
generator.

*Step 2.* $\sum_x q(x)\cdot\tfrac12\left\lvert \frac{p(x)}{q(x)} - 1 \right\rvert
= \tfrac12 \sum_x \lvert p(x) - q(x) \rvert = \mathrm{TV}(P, Q)$.

*Step 3.* $f$ is strictly convex at $1$ in the sense of Definition 3.3 — with subgradient $c = 0$,
$\tilde f(t) = \tfrac12\lvert t-1\rvert \gt 0$ for $t \neq 1$ — so Theorem 4.1 gives
$\mathrm{TV} = 0 \iff P = Q$. It is **not** strictly convex on $(0,\infty)$, which is why
Example 6.3 can preserve it across a non-sufficient channel.

$$
\boxed{f(t) = \tfrac{1}{2}\lvert t - 1 \rvert \Rightarrow D_f(P \parallel Q) = \mathrm{TV}(P, Q)}
$$

**Key takeaway.** Strict convexity *at $1$* and strict convexity *everywhere* are different
hypotheses, and total variation separates them.

In [7]:
f_tv = lambda t: 0.5 * np.abs(t - 1.0)
worst = 0.0
for _ in range(300):
    pa = rng.dirichlet(np.ones(4))
    qa = rng.dirichlet(np.ones(4))
    worst = max(worst, abs(float(np.sum(qa * f_tv(pa / qa))) - tv(pa, qa)))
print(f"worst |D_f - TV| over 300 random 4-outcome pairs = {worst:.3e} = {worst / EPS:.2f} * eps")
assert worst < 10 * EPS

worst |D_f - TV| over 300 random 4-outcome pairs = 2.220e-16 = 1.00 * eps


### Problem L0.7 — Two Gaussians of equal variance

**Statement.** Specialize Theorem 4.6 to $\sigma_1 = \sigma_2 = \sigma$ and evaluate for
$\mu_1 = 1$, $\mu_2 = 0$, $\sigma = 2$.

**Intuition.** With the shapes matched, only the displacement can cost anything.

**Solution.**

*Step 1.* In Theorem 4.6 put $\sigma_1 = \sigma_2 = \sigma$. The term $\ln(\sigma_2/\sigma_1)$
vanishes and $\sigma_1^2/(2\sigma_2^2) = \tfrac12$ cancels the $-\tfrac12$.

*Step 2.* What is left is $D_{\mathrm{KL}} = \dfrac{(\mu_1-\mu_2)^2}{2\sigma^2}$, which is symmetric
in the two means.

*Step 3.* With $\mu_1 - \mu_2 = 1$ and $\sigma = 2$: $1/(2 \cdot 4) = 0.125$ nats.

$$
\boxed{D_{\mathrm{KL}}\bigl(\mathcal{N}(\mu_1,\sigma^2) \parallel \mathcal{N}(\mu_2,\sigma^2)\bigr) = \frac{(\mu_1-\mu_2)^2}{2\sigma^2} = 0.125 \text{ nats here}}
$$

**Key takeaway.** Equal variances make relative entropy a scaled squared distance, and only then;
Example 6.5 shows the asymmetry returning as soon as the variances differ.

In [8]:
print(f"D(N(1,4) || N(0,4)) = {gauss_kl(1, 2, 0, 2):.6f}")
print(f"D(N(0,4) || N(1,4)) = {gauss_kl(0, 2, 1, 2):.6f}")
print(f"(mu1 - mu2)^2 / (2 sigma^2) = {1 ** 2 / (2 * 2 ** 2):.6f}")
assert abs(gauss_kl(1, 2, 0, 2) - 0.125) < 1e-12
assert abs(gauss_kl(1, 2, 0, 2) - gauss_kl(0, 2, 1, 2)) < 1e-12

D(N(1,4) || N(0,4)) = 0.125000
D(N(0,4) || N(1,4)) = 0.125000
(mu1 - mu2)^2 / (2 sigma^2) = 0.125000


## L1 — Foundations

### Problem L1.1 — Nonnegativity of every $f$-divergence

**Statement.** Prove $D_f(P \parallel Q) \ge 0$ for every convex $f$ with $f(1) = 0$, and that
strict convexity at $1$ forces equality only at $P = Q$.

**Intuition.** Convexity plus normalization is the whole argument; the logarithm plays no role.

**Solution.**

*Step 1 (Jensen).* With $X \sim Q$ and $T = p(X)/q(X)$, convexity gives
$D_f(P \parallel Q) = \mathbb{E}_Q\left[f(T)\right] \ge f\!\left(\mathbb{E}_Q[T]\right)$.

*Step 2 (the mean of the ratio).*
$\mathbb{E}_Q[T] = \sum_x q(x)\frac{p(x)}{q(x)} = \sum_x p(x) = 1$.

*Step 3 (normalization).* Hence $D_f(P \parallel Q) \ge f(1) = 0$.

*Step 4 (equality).* Jensen with a function strictly convex at $1$ is tight only if $T$ is
$Q$-almost surely equal to its mean, which is $1$; so $p = q$ on the support of $Q$, and with both
normalized, $P = Q$.

$$
\boxed{D_f(P \parallel Q) \ge f(1) = 0, \qquad \text{equality} \iff P = Q \text{ for } f \text{ strictly convex at } 1}
$$

**Key takeaway.** One line of Jensen covers the whole family. Proof 5.1 gives the sharper
supporting-hyperplane version, which handles the equality case without assuming differentiability.

In [9]:
gens = {"KL": lambda t: t * np.log(t),
        "reverse KL": lambda t: -np.log(t),
        "TV": lambda t: 0.5 * np.abs(t - 1.0),
        "chi-squared": lambda t: (t - 1.0) ** 2,
        "squared Hellinger": lambda t: (np.sqrt(t) - 1.0) ** 2,
        "Jensen-Shannon": lambda t: 0.5 * (t * np.log(t) - (t + 1) * np.log((t + 1) / 2))}
worst_neg = 0.0
for _ in range(400):
    pa = rng.dirichlet(np.ones(4))
    qa = rng.dirichlet(np.ones(4))
    for name, g in gens.items():
        worst_neg = min(worst_neg, float(np.sum(qa * g(pa / qa))))
print(f"most negative value of any D_f over 400 random pairs x 6 generators: {worst_neg:.3e}")
print("value at P = Q for each generator:")
pa = rng.dirichlet(np.ones(4))
for name, g in gens.items():
    print(f"  {name:18s} {float(np.sum(pa * g(pa / pa))):.3e}")
assert worst_neg > -10 * EPS

most negative value of any D_f over 400 random pairs x 6 generators: 0.000e+00
value at P = Q for each generator:
  KL                 0.000e+00
  reverse KL         0.000e+00
  TV                 0.000e+00
  chi-squared        0.000e+00
  squared Hellinger  0.000e+00
  Jensen-Shannon     0.000e+00


### Problem L1.2 — The chain rule on a two-variable example

**Statement.** Let $X \sim \mathrm{Ber}(1/2)$ under both $P$ and $Q$. Under $P$,
$Y \mid X{=}1 \sim \mathrm{Ber}(0.9)$ and $Y \mid X{=}0 \sim \mathrm{Ber}(0.1)$; under $Q$,
$Y \mid X \sim \mathrm{Ber}(0.5)$ regardless. Compute $D_{\mathrm{KL}}(P_{XY} \parallel Q_{XY})$ by
Theorem 4.5 and verify it directly.

**Intuition.** The marginals agree, so all of the divergence has to come from the conditionals.

**Solution.**

*Step 1.* $D_{\mathrm{KL}}(P_X \parallel Q_X) = 0$ because both marginals are $\mathrm{Ber}(1/2)$.

*Step 2.* Each conditional term compares $\mathrm{Ber}(0.9)$ or $\mathrm{Ber}(0.1)$ with the
uniform, which by Problem L0.1 costs $0.531004$ bits in both cases.

*Step 3.* Averaging over $x$: $0 + \tfrac12(0.531004) + \tfrac12(0.531004) = 0.531004$ bits.

*Step 4 (direct).* The joint tables are $P_{XY} = (0.45, 0.05, 0.05, 0.45)$ and
$Q_{XY} = (0.25, 0.25, 0.25, 0.25)$, so
$\sum p \log_2 \frac{p}{0.25} = 2 - H(P_{XY}) = 2 - 1.468996 = 0.531004$ bits.

$$
\boxed{D_{\mathrm{KL}}(P_{XY} \parallel Q_{XY}) = 0.531004 \text{ bits, all of it from the conditionals}}
$$

**Key takeaway.** Two models can share every marginal and still be far apart; only the joint
divergence sees the dependence structure.

In [10]:
P_joint = np.array([0.45, 0.05, 0.05, 0.45])
Q_joint = np.full(4, 0.25)
cond = 0.5 * kl(np.array([0.9, 0.1]), Q) + 0.5 * kl(np.array([0.1, 0.9]), Q)
print(f"direct     : {kl(P_joint, Q_joint) * BITS:.6f} bits")
print(f"chain rule : 0 + {cond * BITS:.6f} bits")
print(f"2 - H(P_XY): {2 - stats.entropy(P_joint, base=2):.6f} bits")
assert abs(kl(P_joint, Q_joint) - cond) < 1e-12
assert abs(kl(P_joint, Q_joint) * BITS - 0.531004) < 5e-7

direct     : 0.531004 bits
chain rule : 0 + 0.531004 bits
2 - H(P_XY): 0.531004 bits


### Problem L1.3 — The variational-autoencoder divergence in closed form

**Statement.** Derive
$D_{\mathrm{KL}}\bigl(\mathcal{N}(\mu, \operatorname{diag}(\sigma^2)) \parallel \mathcal{N}(0, I)\bigr)$
in $d$ dimensions and evaluate it for $d = 2$ with $\mu = (1, 0)$ and $\sigma = (0.5, 1)$.

**Intuition.** Both distributions factor across coordinates, so Theorem 4.5 turns the problem into
$d$ copies of Theorem 4.6.

**Solution.**

*Step 1.* Additivity over independent factors reduces the $d$-dimensional divergence to a sum of
one-dimensional terms.

*Step 2.* Theorem 4.6 with $\mu_2 = 0$, $\sigma_2 = 1$ gives
$D_j = -\ln\sigma_j + \frac{\sigma_j^2 + \mu_j^2}{2} - \frac12
= \frac12\left(\mu_j^2 + \sigma_j^2 - 1 - \ln\sigma_j^2\right)$.

*Step 3.* Summing, $D_{\mathrm{KL}} = \frac12\sum_{j=1}^{d}\left(\mu_j^2 + \sigma_j^2 - 1 - \ln\sigma_j^2\right)$.

*Step 4 (numbers).* Coordinate 1: $\tfrac12(1 + 0.25 - 1 + 1.386294) = 0.818147$. Coordinate 2:
$\tfrac12(0 + 1 - 1 - 0) = 0$, an exactly collapsed dimension.

$$
\boxed{D_{\mathrm{KL}} = \tfrac{1}{2}\sum_j\left(\mu_j^2 + \sigma_j^2 - 1 - \ln\sigma_j^2\right) = 0.818147 \text{ nats here}}
$$

**Key takeaway.** Each latent dimension contributes independently and nonnegatively, and a
dimension at $(\mu_j, \sigma_j) = (0,1)$ contributes exactly zero — the analytic signature of
posterior collapse.

In [11]:
mu = np.array([1.0, 0.0])
sig = np.array([0.5, 1.0])
terms = 0.5 * (mu ** 2 + sig ** 2 - 1.0 - np.log(sig ** 2))
print("per-coordinate KL:", terms)
print(f"total = {terms.sum():.6f} nats")
print("check against Theorem 4.6 term by term:",
      np.array([gauss_kl(m, s, 0.0, 1.0) for m, s in zip(mu, sig)]))
assert np.allclose(terms, [gauss_kl(m, s, 0.0, 1.0) for m, s in zip(mu, sig)])
assert abs(terms.sum() - 0.818147) < 5e-7

per-coordinate KL: [0.8181 0.    ]
total = 0.818147 nats
check against Theorem 4.6 term by term: [0.8181 0.    ]


### Problem L1.4 — Additivity over independent repetitions

**Statement.** Show $D_{\mathrm{KL}}(P^{\otimes n} \parallel Q^{\otimes n}) = n\,D_{\mathrm{KL}}(P \parallel Q)$
and state the consequence for hypothesis testing.

**Intuition.** Independent observations contribute independent evidence, and evidence is a
logarithm, hence additive.

**Solution.**

*Step 1.* Under independence the log-ratio is a sum:
$\ln\frac{\prod_i p(x_i)}{\prod_i q(x_i)} = \sum_i \ln\frac{p(x_i)}{q(x_i)}$.

*Step 2.* Take the expectation under $P^{\otimes n}$; linearity and identical marginals give
$n\,D_{\mathrm{KL}}(P \parallel Q)$. This is Theorem 4.5 with every conditional equal to its
marginal.

*Step 3 (consequence).* Evidence grows linearly in $n$, so error probabilities decay
exponentially: by Problem L3.3 the optimal test's type-II error satisfies
$\tfrac1n\ln\beta_n \to -D_{\mathrm{KL}}(P \parallel Q)$.

$$
\boxed{D_{\mathrm{KL}}(P^{\otimes n} \parallel Q^{\otimes n}) = n\,D_{\mathrm{KL}}(P \parallel Q), \qquad \beta_n = e^{-n D_{\mathrm{KL}} (1 + o(1))}}
$$

**Key takeaway.** Relative entropy is an information *rate* per observation, which is why it and
not total variation indexes sample complexity.

In [12]:
Pn, Qn = np.array([0.7, 0.3]), np.array([0.4, 0.6])
base_kl = kl(Pn, Qn)
print(f"D(P||Q) = {base_kl:.8f}")
for n in (1, 2, 3, 5, 8):
    Pk = Pn.copy()
    Qk = Qn.copy()
    for _ in range(n - 1):
        Pk = np.outer(Pk, Pn).ravel()
        Qk = np.outer(Qk, Qn).ravel()
    print(f"  n = {n}: D(P^n || Q^n) = {kl(Pk, Qk):.8f}   n * D = {n * base_kl:.8f}")
    assert abs(kl(Pk, Qk) - n * base_kl) < 1e-12

D(P||Q) = 0.18378690
  n = 1: D(P^n || Q^n) = 0.18378690   n * D = 0.18378690
  n = 2: D(P^n || Q^n) = 0.36757379   n * D = 0.36757379
  n = 3: D(P^n || Q^n) = 0.55136069   n * D = 0.55136069
  n = 5: D(P^n || Q^n) = 0.91893449   n * D = 0.91893449
  n = 8: D(P^n || Q^n) = 1.47029518   n * D = 1.47029518


### Problem L1.5 — $\chi^2$ dominates relative entropy

**Statement.** Prove
$D_{\mathrm{KL}}(P \parallel Q) \le \ln\left(1 + \chi^2(P \parallel Q)\right) \le \chi^2(P \parallel Q)$.

**Intuition.** Jensen applied to the concave logarithm turns the mean of a log into the log of a
mean, and that mean is exactly $1 + \chi^2$.

**Solution.**

*Step 1.*
$\chi^2(P \parallel Q) = \sum_x \frac{p(x)^2 - 2p(x)q(x) + q(x)^2}{q(x)} = \sum_x \frac{p(x)^2}{q(x)} - 1$,
so $\mathbb{E}_P\left[\frac{p(X)}{q(X)}\right] = 1 + \chi^2$.

*Step 2.* With $X \sim P$ and $T = p(X)/q(X)$, concavity of $\ln$ gives
$D_{\mathrm{KL}}(P \parallel Q) = \mathbb{E}_P[\ln T] \le \ln \mathbb{E}_P[T] = \ln(1 + \chi^2)$.

*Step 3.* The tangent-line bound $\ln(1+u) \le u$ finishes it.

$$
\boxed{D_{\mathrm{KL}}(P \parallel Q) \le \ln\left(1 + \chi^2(P \parallel Q)\right) \le \chi^2(P \parallel Q)}
$$

**Key takeaway.** $\chi^2$ is the harshest of the standard divergences, so bounding it — as
importance-sampling variance analyses do — automatically bounds relative entropy.

In [13]:
worst_gap = np.inf
for _ in range(2000):
    pa = rng.dirichlet(np.ones(4))
    qa = rng.dirichlet(np.ones(4))
    c2 = chi2(pa, qa)
    worst_gap = min(worst_gap, np.log1p(c2) - kl(pa, qa))
    assert kl(pa, qa) <= np.log1p(c2) + 1e-12 <= c2 + 1e-12
print(f"smallest observed  ln(1 + chi2) - KL  over 2000 pairs = {worst_gap:.3e}  (never negative)")
print(f"on the Example 6.3 pair: KL = {kl(Pa, Qa):.6f} <= ln(1 + chi2) = "
      f"{np.log1p(chi2(Pa, Qa)):.6f} <= chi2 = {chi2(Pa, Qa):.6f}")

smallest observed  ln(1 + chi2) - KL  over 2000 pairs = 5.866e-03  (never negative)
on the Example 6.3 pair: KL = 0.675806 <= ln(1 + chi2) = 1.100990 <= chi2 = 2.007143


### Problem L1.6 — Jensen-Shannon is bounded by $\ln 2$

**Statement.** Prove $0 \le \mathrm{JS}(P, Q) \le \ln 2$, show the upper bound is attained exactly
when $P$ and $Q$ have disjoint supports, and say why it matters for adversarial training.

**Intuition.** The mixture $M$ dominates each argument by a factor of at most two, so each
log-ratio is capped at $\ln 2$.

**Solution.**

*Step 1 (upper bound).* $m(x) = \tfrac12(p(x) + q(x)) \ge \tfrac12 p(x)$, so
$\ln\frac{p(x)}{m(x)} \le \ln 2$ pointwise and $D_{\mathrm{KL}}(P \parallel M) \le \ln 2$; likewise
for $Q$. Averaging gives $\mathrm{JS} \le \ln 2$.

*Step 2 (equality).* Equality needs $m = \tfrac12 p$ on the support of $P$, that is $q = 0$ there,
and symmetrically: disjoint supports. Then each divergence is exactly $\ln 2$.

*Step 3 (nonnegativity).* Equivalently
$\mathrm{JS}(P,Q) = H(M) - \tfrac12 H(P) - \tfrac12 H(Q) \ge 0$ by concavity of entropy.

*Step 4 (consequence).* A generator whose samples lie off the data manifold sits at the flat
maximum $\mathrm{JS} = \ln 2$, where the objective is constant in its parameters and the gradient
vanishes.

$$
\boxed{0 \le \mathrm{JS}(P, Q) \le \ln 2, \text{ with } \ln 2 \text{ exactly for disjoint supports}}
$$

**Key takeaway.** Boundedness keeps $\mathrm{JS}$ finite where $D_{\mathrm{KL}}$ explodes, and
flattens the objective exactly where a generator most needs a gradient.

In [14]:
Pd = np.array([0.5, 0.5, 0.0, 0.0])
Qd = np.array([0.0, 0.0, 0.3, 0.7])
print(f"disjoint supports : JS = {js(Pd, Qd):.12f}   ln 2 = {np.log(2.0):.12f}")
worst_js = 0.0
for _ in range(2000):
    pa = rng.dirichlet(np.ones(4))
    qa = rng.dirichlet(np.ones(4))
    worst_js = max(worst_js, js(pa, qa))
    assert -1e-15 <= js(pa, qa) <= np.log(2.0) + 1e-12
print(f"largest JS over 2000 overlapping random pairs = {worst_js:.6f}  (below ln 2)")
Md = 0.5 * (Pd + Qd)
ent = lambda v: float(-np.sum(v[v > 0] * np.log(v[v > 0])))
print(f"entropy form: H(M) - (H(P)+H(Q))/2 = {ent(Md) - 0.5 * ent(Pd) - 0.5 * ent(Qd):.12f}")
assert abs(js(Pd, Qd) - np.log(2.0)) < 1e-12

disjoint supports : JS = 0.693147180560   ln 2 = 0.693147180560


largest JS over 2000 overlapping random pairs = 0.535943  (below ln 2)
entropy form: H(M) - (H(P)+H(Q))/2 = 0.693147180560


### Problem L1.7 — The log-sum inequality, and Gibbs from it

**Statement.** Prove Theorem 4.2 and deduce $D_{\mathrm{KL}}(P \parallel Q) \ge 0$ from it in one
line.

**Intuition.** Pooling two cells replaces two likelihood ratios by their weighted average, and
$t \ln t$ is convex.

**Solution.**

*Step 1.* Put $b = \sum_i b_i$ and $\lambda_i = b_i/b$, a probability vector.

*Step 2.* $g(t) = t\ln t$ is convex, so Jensen at the points $t_i = a_i/b_i$ gives
$\sum_i \lambda_i g(a_i/b_i) \ge g\!\left(\sum_i \lambda_i a_i/b_i\right) = g\!\left(\tfrac{\sum_i a_i}{b}\right)$.

*Step 3.* Multiplying by $b$ turns the left side into $\sum_i a_i\ln\frac{a_i}{b_i}$ and the right
side into $\left(\sum_i a_i\right)\ln\frac{\sum_i a_i}{\sum_i b_i}$.

*Step 4 (Gibbs).* Apply it with $a_i = p(x_i)$ and $b_i = q(x_i)$ over the whole alphabet: the
right-hand side is $1 \cdot \ln\frac{1}{1} = 0$, so $D_{\mathrm{KL}}(P \parallel Q) \ge 0$.

$$
\boxed{\sum_i a_i \ln\frac{a_i}{b_i} \ge \left(\sum_i a_i\right)\ln\frac{\sum_i a_i}{\sum_i b_i}, \qquad \text{equality} \iff a_i/b_i \text{ constant}}
$$

**Key takeaway.** The log-sum inequality is the data-processing inequality for the single channel
that merges two symbols; Example 6.3 measures the two as the same number.

In [15]:
a_vec = np.array([0.5, 0.3])
b_vec = np.array([0.1, 0.2])
lhs = float(np.sum(a_vec * np.log(a_vec / b_vec)))
rhs = float(a_vec.sum() * np.log(a_vec.sum() / b_vec.sum()))
print(f"log-sum: lhs = {lhs:.7f}  rhs = {rhs:.7f}  gap = {lhs - rhs:.7f}")
print("equality case, a/b constant:", end=" ")
a_eq, b_eq = np.array([0.4, 0.8]), np.array([0.1, 0.2])
print(f"gap = {float(np.sum(a_eq * np.log(a_eq / b_eq))) - float(a_eq.sum() * np.log(a_eq.sum() / b_eq.sum())):.3e}")
worst = np.inf
for _ in range(2000):
    av = rng.random(5)
    bv = rng.random(5) + 1e-3
    worst = min(worst, float(np.sum(av * np.log(av / bv))) - float(av.sum() * np.log(av.sum() / bv.sum())))
print(f"smallest gap over 2000 random unnormalized vectors = {worst:.3e}  (never negative)")
assert worst > -1e-12

log-sum: lhs = 0.9263585  rhs = 0.7846634  gap = 0.1416951
equality case, a/b constant: gap = -4.441e-16
smallest gap over 2000 random unnormalized vectors = 2.127e-02  (never negative)


### Problem L1.8 — Hellinger sandwiches total variation

**Statement.** With $H^2(P,Q) = \sum_x (\sqrt{p(x)} - \sqrt{q(x)})^2$, prove

$$
\tfrac{1}{2} H^2(P,Q) \; \le \; \mathrm{TV}(P, Q) \; \le \; H(P,Q)\sqrt{1 - \tfrac{1}{4}H^2(P,Q)} .
$$

**Intuition.** Factor $p - q = (\sqrt p - \sqrt q)(\sqrt p + \sqrt q)$; the first factor is Hellinger
and the second is bounded above and below.

**Solution.**

*Step 1 (lower bound).* $\lvert \sqrt{p} + \sqrt{q} \rvert \ge \lvert \sqrt{p} - \sqrt{q} \rvert$, so

$$
\mathrm{TV} = \tfrac12 \sum_x \lvert \sqrt{p} - \sqrt{q}\rvert \lvert \sqrt{p} + \sqrt{q}\rvert
\;\ge\; \tfrac12 \sum_x (\sqrt{p} - \sqrt{q})^2 = \tfrac12 H^2 .
$$

*Step 2 (upper bound).* Cauchy-Schwarz on the same factorization gives

$$
2\,\mathrm{TV} \le \sqrt{\sum_x (\sqrt p - \sqrt q)^2}\;\sqrt{\sum_x (\sqrt p + \sqrt q)^2} = H \sqrt{4 - H^2},
$$

using $\sum_x (\sqrt p + \sqrt q)^2 = 2 + 2\sum_x\sqrt{pq} = 4 - H^2$.

*Step 3.* Dividing by $2$ turns $H\sqrt{4 - H^2}/2$ into $H\sqrt{1 - H^2/4}$.

$$
\boxed{\tfrac{1}{2} H^2 \;\le\; \mathrm{TV} \;\le\; H\sqrt{1 - \tfrac{1}{4}H^2}}
$$

**Key takeaway.** Hellinger and total variation are equivalent up to squaring, so either can stand
in for the other in a rate; relative entropy is not equivalent to either, since Section 7.4 makes
it infinite at vanishing $\mathrm{TV}$.

In [16]:
worst_lo = worst_hi = np.inf
for _ in range(4000):
    pa = rng.dirichlet(np.ones(4))
    qa = rng.dirichlet(np.ones(4))
    h2 = hellinger2(pa, qa)
    h = np.sqrt(h2)
    worst_lo = min(worst_lo, tv(pa, qa) - 0.5 * h2)
    worst_hi = min(worst_hi, h * np.sqrt(1 - h2 / 4) - tv(pa, qa))
print(f"smallest  TV - H^2/2                 over 4000 pairs = {worst_lo:.3e}")
print(f"smallest  H sqrt(1 - H^2/4) - TV     over 4000 pairs = {worst_hi:.3e}")
print(f"Example 6.3 pair: H^2/2 = {0.5 * hellinger2(Pa, Qa):.6f} <= TV = {tv(Pa, Qa):.6f}"
      f" <= {np.sqrt(hellinger2(Pa, Qa)) * np.sqrt(1 - hellinger2(Pa, Qa) / 4):.6f}")
assert worst_lo > -1e-12 and worst_hi > -1e-12

smallest  TV - H^2/2                 over 4000 pairs = 2.296e-02
smallest  H sqrt(1 - H^2/4) - TV     over 4000 pairs = 2.433e-04
Example 6.3 pair: H^2/2 = 0.157278 <= TV = 0.500000 <= 0.538350


### Problem L1.9 — Squared Hellinger sits below relative entropy

**Statement.** Prove $H^2(P, Q) \le D_{\mathrm{KL}}(P \parallel Q)$.

**Intuition.** The Bhattacharyya coefficient is an expectation of a square root, and Jensen turns
its logarithm into half a relative entropy.

**Solution.**

*Step 1.* Write $\mathrm{BC}(P,Q) = \sum_x \sqrt{p(x)q(x)}$, so that
$H^2 = \sum_x (p + q - 2\sqrt{pq}) = 2 - 2\,\mathrm{BC}$.

*Step 2.* $\mathrm{BC} = \mathbb{E}_P\left[\sqrt{q(X)/p(X)}\right]$, and concavity of $\ln$ gives

$$
\ln \mathrm{BC} \;\ge\; \mathbb{E}_P\left[\ln\sqrt{\tfrac{q(X)}{p(X)}}\right] = -\tfrac{1}{2} D_{\mathrm{KL}}(P \parallel Q),
$$

hence $\mathrm{BC} \ge e^{-D_{\mathrm{KL}}/2}$.

*Step 3.* Therefore
$H^2 = 2(1 - \mathrm{BC}) \le 2\left(1 - e^{-D_{\mathrm{KL}}/2}\right) \le D_{\mathrm{KL}}$, the last
step by $1 - e^{-u} \le u$ with $u = D_{\mathrm{KL}}/2$.

$$
\boxed{H^2(P,Q) \;\le\; 2\left(1 - e^{-D_{\mathrm{KL}}(P \parallel Q)/2}\right) \;\le\; D_{\mathrm{KL}}(P \parallel Q)}
$$

**Key takeaway.** Combined with Problems L1.5 and L1.8 this fixes the ordering
$\tfrac{1}{2}H^2 \le \mathrm{TV}$, $H^2 \le D_{\mathrm{KL}} \le \chi^2$; Section 7.6 checks the
middle chain numerically at $a = 0.9$.

In [17]:
worst_a = worst_b = np.inf
for _ in range(4000):
    pa = rng.dirichlet(np.ones(4))
    qa = rng.dirichlet(np.ones(4))
    d = kl(pa, qa)
    h2 = hellinger2(pa, qa)
    worst_a = min(worst_a, 2 * (1 - np.exp(-d / 2)) - h2)
    worst_b = min(worst_b, d - h2)
print(f"smallest  2(1 - exp(-KL/2)) - H^2  over 4000 pairs = {worst_a:.3e}")
print(f"smallest  KL - H^2                 over 4000 pairs = {worst_b:.3e}")
Ph = np.array([0.9, 0.1])
print(f"at Ber(0.9) vs Ber(0.5): H^2 = {hellinger2(Ph, Q):.6f} <= KL = {kl(Ph, Q):.6f}"
      f" <= chi2 = {chi2(Ph, Q):.6f}")
assert worst_a > -1e-12 and worst_b > -1e-12

smallest  2(1 - exp(-KL/2)) - H^2  over 4000 pairs = 1.031e-03
smallest  KL - H^2                 over 4000 pairs = 1.032e-03
at Ber(0.9) vs Ber(0.5): H^2 = 0.211146 <= KL = 0.368064 <= chi2 = 0.640000


## L2 — Applications (AI/ML and Physics)

Six machine-learning problems and three physics problems. The physics tier is Problems L2.7 to
L2.9: Landauer's bound, the Gibbs state as a free-energy minimizer, and two Maxwell-Boltzmann
gases at different temperatures.

### Problem L2.1 — Reading a variational autoencoder's KL budget

**Statement.** A variational autoencoder with a $64$-dimensional latent reports an average KL term
of $12$ nats. Under a diagonal Gaussian posterior, estimate how many latent dimensions are active
if an active dimension carries about $1$ nat, and say what changes at $\beta = 4$.

**Intuition.** The KL term is a sum of nonnegative per-dimension rates, so the total is a budget
that the active dimensions split.

**Solution.**

*Step 1.* By Problem L1.3, $D_{\mathrm{KL}} = \sum_{j=1}^{64} D_j$ with $D_j \ge 0$ and $D_j = 0$
exactly at $(\mu_j, \sigma_j) = (0, 1)$.

*Step 2.* If active dimensions contribute about $1$ nat and collapsed ones about $0$, then
$12 \approx n_{\text{active}} \times 1$, so roughly $12$ of $64$ dimensions carry information and
$81$ percent of the latent capacity is unused.

*Step 3.* The $\beta$-VAE objective $\mathbb{E}_q[\ln p(x \mid z)] - \beta D_{\mathrm{KL}}$ prices
each nat of latent code at $\beta$ nats of reconstruction. Raising $\beta$ from $1$ to $4$
quadruples that price, so any dimension whose reconstruction payoff is under $4$ nats per nat of
rate collapses: the budget falls and $n_{\text{active}}$ shrinks.

$$
\boxed{n_{\text{active}} \approx D_{\mathrm{KL}} / (\text{nats per active dimension}) \approx 12 \text{ of } 64; \quad \text{larger } \beta \Rightarrow \text{fewer active dimensions}}
$$

**Key takeaway.** The KL term is a rate in nats per sample; monitoring it per dimension is the
standard diagnostic for posterior collapse and for tuning $\beta$.

In [18]:
d_lat = 64
budget = 12.0
mu_a, sig_a = 1.4142135623730951, 1.0        # one active dimension carrying about 1 nat
per_active = 0.5 * (mu_a ** 2 + sig_a ** 2 - 1 - np.log(sig_a ** 2))
n_active = int(round(budget / per_active))
print(f"nats per active dimension (mu = sqrt 2, sigma = 1) : {per_active:.6f}")
print(f"active dimensions implied by a {budget:.0f}-nat budget : {n_active} of {d_lat}")
print(f"fraction of latent capacity unused                 : {1 - n_active / d_lat:.4f}")
mu_vec = np.zeros(d_lat)
sig_vec = np.ones(d_lat)
mu_vec[:n_active] = mu_a
total = float(np.sum(0.5 * (mu_vec ** 2 + sig_vec ** 2 - 1 - np.log(sig_vec ** 2))))
print(f"rebuilt total KL over all 64 dimensions            : {total:.6f} nats")
assert n_active == 12 and abs(total - budget) < 1e-9

nats per active dimension (mu = sqrt 2, sigma = 1) : 1.000000
active dimensions implied by a 12-nat budget : 12 of 64
fraction of latent capacity unused                 : 0.8125
rebuilt total KL over all 64 dimensions            : 12.000000 nats


### Problem L2.2 — The $k_1$, $k_2$, $k_3$ estimators used in policy optimization

**Statement.** To monitor $D_{\mathrm{KL}}(\pi \parallel \pi_{\mathrm{ref}})$ from samples
$y \sim \pi$, define $r = \pi_{\mathrm{ref}}(y)/\pi(y)$ and

$$
k_1 = -\ln r, \qquad k_2 = \tfrac12 (\ln r)^2, \qquad k_3 = (r - 1) - \ln r .
$$

Analyse bias, sign and variance, and decide when $k_3$ is preferable to $k_1$.

**Intuition.** $k_3$ adds the zero-mean control variate $r - 1$ to $k_1$. A control variate helps
only when it is negatively correlated with what it corrects.

**Solution.**

*Step 1 ($k_1$).* $\mathbb{E}_\pi[-\ln r] = \mathbb{E}_\pi\left[\ln\frac{\pi(y)}{\pi_{\mathrm{ref}}(y)}\right]
= D_{\mathrm{KL}}(\pi \parallel \pi_{\mathrm{ref}})$: unbiased. Individual samples are negative
whenever $\pi_{\mathrm{ref}}(y) \gt \pi(y)$, so short runs can report a negative "divergence".

*Step 2 ($k_3$ is also unbiased and nonnegative).* $\mathbb{E}_\pi[r] = \sum_y \pi(y)\frac{\pi_{\mathrm{ref}}(y)}{\pi(y)} = 1$,
so $\mathbb{E}_\pi[k_3] = 0 + D_{\mathrm{KL}}(\pi \parallel \pi_{\mathrm{ref}})$; and $k_3 \ge 0$
pointwise because $r - 1 \ge \ln r$ for all $r \gt 0$.

*Step 3 (variance — the claim that must be stated carefully).* Since $k_3 = k_1 + (r-1)$,

$$
\operatorname{Var}(k_3) = \operatorname{Var}(k_1) + \operatorname{Var}(r) + 2\operatorname{Cov}(k_1, r),
$$

so $k_3$ beats $k_1$ **exactly when** $\operatorname{Var}(r) + 2\operatorname{Cov}(-\ln r, r) \lt 0$.
This is not automatic. Writing $r = 1 + \delta$ with $\delta$ small gives $k_1 = -\delta + O(\delta^2)$
and $k_3 = \tfrac12\delta^2 + O(\delta^3)$, so in the near-reference regime — the RLHF regime, where
$\pi$ stays close to $\pi_{\mathrm{ref}}$ — the variance of $k_3$ is smaller by two orders in
$\delta$. When the ratio is heavy-tailed, $\operatorname{Var}(r)$ dominates and $k_3$ is **worse**.

*Step 4 (counterexample).* Take $\pi = (0.009, 0.991)$ and $\pi_{\mathrm{ref}} = (0.9, 0.1)$, so
$r = (100, 0.100908)$ and $\mathbb{E}_\pi[r] = 1$ exactly. Then
$\operatorname{Var}(k_1) = 0.424475$ while $\operatorname{Var}(k_3) = 77.141060$: the control variate
makes the estimator $182$ times worse.

*Step 5 ($k_2$).* $k_2 = \tfrac12(\ln r)^2$ is nonnegative and low variance but **biased**: it is
correct only to second order in $\ln r$, and on the counterexample its bias is $0.470479$ nats.

*Step 6 (the optimal coefficient).* The best control variate of this form is
$k_c = -\ln r + c(r-1)$ with
$c^{\star} = -\operatorname{Cov}(-\ln r,\, r)/\operatorname{Var}(r)$, which is unbiased for every $c$
and never worse than either $k_1$ ($c = 0$) or $k_3$ ($c = 1$).

$$
\boxed{k_1, k_3 \text{ unbiased}; \; k_3 \ge 0 \text{ and lower variance only when } \operatorname{Var}(r) + 2\operatorname{Cov}(-\ln r, r) \lt 0; \; k_2 \text{ biased}}
$$

**Key takeaway.** $k_3 = r - 1 - \ln r$ is the generator of $D_{\mathrm{KL}}$ normalized to
$f(1) = f'(1) = 0$ (Problem L0.3), which is why it is nonnegative — but nonnegativity is not
variance reduction, and the two regimes below separate them by six orders of magnitude.

In [19]:
def estimator_stats(pi, pi_ref, label):
    r = pi_ref / pi
    k1, k2, k3 = -np.log(r), 0.5 * np.log(r) ** 2, (r - 1.0) - np.log(r)
    D = kl(pi, pi_ref)
    mean = lambda v: float(pi @ v)
    var = lambda v: float(pi @ (v - mean(v)) ** 2)
    cov = float(pi @ ((k1 - mean(k1)) * (r - 1.0)))
    c_star = -cov / var(r)
    kc = k1 + c_star * (r - 1.0)
    print(f"{label}:  pi = {pi}   pi_ref = {pi_ref}   E[r] = {float(pi @ r):.12f}")
    print(f"   true D_KL = {D:.8f}")
    for nm, v in (("k1", k1), ("k2", k2), ("k3", k3), ("k*", kc)):
        print(f"   {nm}: mean = {mean(v):.8f}  bias = {mean(v) - D:+.3e}  var = {var(v):.6e}")
    print(f"   optimal coefficient c* = {c_star:.6f}   "
          f"Var(r) + 2Cov(-ln r, r) = {var(r) + 2 * cov:.6e}")
    return var(k1), var(k3)


v1_far, v3_far = estimator_stats(np.array([0.009, 0.991]), np.array([0.9, 0.1]),
                                 "heavy-tailed ratio")
print()
v1_near, v3_near = estimator_stats(np.array([0.5, 0.5]), np.array([0.525, 0.475]),
                                   "near-reference (the RLHF regime)")
print(f"\nvariance ratio Var(k3)/Var(k1):  heavy-tailed {v3_far / v1_far:10.2f}"
      f"   near-reference {v3_near / v1_near:.3e}")
assert v3_far > v1_far and v3_near < v1_near

heavy-tailed ratio:  pi = [0.009 0.991]   pi_ref = [0.9 0.1]   E[r] = 1.000000000000
   true D_KL = 2.23145592
   k1: mean = 2.23145592  bias = +0.000e+00  var = 4.244754e-01
   k2: mean = 2.70193545  bias = +4.705e-01  var = 5.670582e-01
   k3: mean = 2.23145592  bias = +0.000e+00  var = 7.714106e+01
   k*: mean = 2.23145592  bias = +0.000e+00  var = 2.839899e-32
   optimal coefficient c* = 0.069057   Var(r) + 2Cov(-ln r, r) = 7.671658e+01

near-reference (the RLHF regime):  pi = [0.5 0.5]   pi_ref = [0.525 0.475]   E[r] = 1.000000000000
   true D_KL = 0.00125157
   k1: mean = 0.00125157  bias = +5.204e-17  var = 2.504175e-03
   k2: mean = 0.00125287  bias = +1.305e-06  var = 3.922577e-09
   k3: mean = 0.00125157  bias = +5.204e-17  var = 1.741333e-09
   k*: mean = 0.00125157  bias = +5.204e-17  var = 4.814825e-35
   optimal coefficient c* = 1.000835   Var(r) + 2Cov(-ln r, r) = -2.504173e-03

variance ratio Var(k3)/Var(k1):  heavy-tailed     181.73   near-reference 6.954e-07


### Problem L2.3 — Fitting one Gaussian to two modes

**Statement.** Let $p = \tfrac12\mathcal{N}(-4,1) + \tfrac12\mathcal{N}(4,1)$. Find the minimizer of
$D_{\mathrm{KL}}(p \parallel q)$ over $q = \mathcal{N}(\mu,\sigma^2)$ and describe the minimizers of
$D_{\mathrm{KL}}(q \parallel p)$.

**Intuition.** Forward KL integrates against $p$ and cannot leave any of $p$'s mass uncovered;
reverse KL integrates against $q$ and never inspects what $q$ ignores.

**Solution.**

*Step 1 (forward).* By Theorem 4.8(a), minimizing $D_{\mathrm{KL}}(p \parallel q)$ maximizes
$\mathbb{E}_p[\ln q]$, which is moment matching:
$\mu^{\star} = \mathbb{E}_p[X] = 0$ and $(\sigma^{\star})^2 = \operatorname{Var}_p(X) = 1 + 16 = 17$.

*Step 2 (reverse).* By Theorem 4.8(b), $D_{\mathrm{KL}}(\mathcal{N}(4,1) \parallel p) \le \ln 2$
while $D_{\mathrm{KL}}(\mathcal{N}(0,17) \parallel p) \ge 1.424344$, so the single-mode fit strictly
wins. Numerically the two values are $0.693054$ and $2.097617$.

*Step 3 (structure of the reverse objective).* By symmetry there are two minimizers,
$\mathcal{N}(\pm 4, 1)$, with $\mu = 0$ a local maximum along the mean: the barrier there is
$5.420989$ nats, plotted in Section 2 of the theory notebook.

$$
\boxed{\text{forward: } \mathcal{N}(0, 17) \text{ uniquely}; \qquad \text{reverse: } \mathcal{N}(\pm 4, 1), \text{ two symmetric minima}}
$$

**Key takeaway.** The direction is a modelling decision: maximum likelihood and distillation never
omit data but overgeneralize, while variational inference and KL-regularized policies are sharp and
can silently drop a mode.

In [20]:
c_sep = 4.0


def log_p_mix(x, c=4.0):
    return np.logaddexp(stats.norm.logpdf(x, -c, 1.0) + np.log(0.5),
                        stats.norm.logpdf(x, c, 1.0) + np.log(0.5))


def rev_kl(mu, sigma, c=4.0):
    g = lambda x: stats.norm.pdf(x, mu, sigma) * (stats.norm.logpdf(x, mu, sigma)
                                                  - log_p_mix(x, c))
    return integrate.quad(g, mu - 12 * sigma, mu + 12 * sigma, limit=500)[0]


def fwd_kl(mu, sigma, c=4.0):
    g = lambda x: np.exp(log_p_mix(x, c)) * (log_p_mix(x, c) - stats.norm.logpdf(x, mu, sigma))
    return integrate.quad(g, -40.0, 40.0, limit=500)[0]


grid_mu = np.linspace(-6, 6, 25)
grid_s2 = np.linspace(1.0, 30.0, 30)
best_f = min(((fwd_kl(m, np.sqrt(s2)), m, s2) for m in grid_mu for s2 in grid_s2))
print(f"forward-KL grid minimum : mu = {best_f[1]:.3f}, sigma^2 = {best_f[2]:.3f},"
      f" value = {best_f[0]:.6f}   (theory: mu = 0, sigma^2 = 17)")
print(f"forward : D(p||N(0,17)) = {fwd_kl(0.0, np.sqrt(17)):.6f}   "
      f"D(p||N(4,1)) = {fwd_kl(4.0, 1.0):.6f}")
print(f"reverse : D(N(4,1)||p)  = {rev_kl(4.0, 1.0):.6f}   "
      f"D(N(0,17)||p) = {rev_kl(0.0, np.sqrt(17)):.6f}")
print(f"barrier at mu = 0, sigma = 1: {rev_kl(0.0, 1.0):.6f} nats;   ln 2 = {np.log(2.0):.6f}")
assert abs(best_f[1]) < 1e-9 and abs(best_f[2] - 17.0) < 1.1
assert rev_kl(4.0, 1.0) < rev_kl(0.0, np.sqrt(17))
assert fwd_kl(0.0, np.sqrt(17)) < fwd_kl(4.0, 1.0)

forward-KL grid minimum : mu = 0.000, sigma^2 = 17.000, value = 0.723553   (theory: mu = 0, sigma^2 = 17)
forward : D(p||N(0,17)) = 0.723553   D(p||N(4,1)) = 15.306946
reverse : D(N(4,1)||p)  = 0.693054   D(N(0,17)||p) = 2.097617
barrier at mu = 0, sigma = 1: 5.420989 nats;   ln 2 = 0.693147


### Problem L2.4 — The adversarial objective is a Jensen-Shannon divergence

**Statement.** For $V(D, G) = \mathbb{E}_{x \sim p}[\ln D(x)] + \mathbb{E}_{x \sim g}[\ln(1 - D(x))]$,
find the optimal discriminator and show $\max_D V = 2\,\mathrm{JS}(p, g) - 2\ln 2$.

**Intuition.** For each $x$ the discriminator solves a one-dimensional problem whose answer is the
posterior probability that $x$ came from $p$.

**Solution.**

*Step 1 (pointwise).* $V = \int \left(p(x)\ln D(x) + g(x)\ln(1 - D(x))\right)dx$, and
$a\ln D + b\ln(1-D)$ is maximized at $D = a/(a+b)$. Hence $D^{\star}(x) = \frac{p(x)}{p(x)+g(x)}$.

*Step 2 (substitute).* With $m = \tfrac12(p+g)$ we have $D^{\star} = \frac{p}{2m}$ and
$1 - D^{\star} = \frac{g}{2m}$, so

$$
V(D^{\star}, G) = \mathbb{E}_p\left[\ln\tfrac{p}{2m}\right] + \mathbb{E}_g\left[\ln\tfrac{g}{2m}\right]
= D_{\mathrm{KL}}(p \parallel m) + D_{\mathrm{KL}}(g \parallel m) - 2\ln 2 .
$$

*Step 3 (identify).* The two divergences sum to $2\,\mathrm{JS}(p,g)$ by Definition 3.5, so
$V(D^{\star}, G) = 2\,\mathrm{JS}(p,g) - 2\ln 2$, minimized at $g = p$ with value
$-2\ln 2 = -1.386294$.

$$
\boxed{D^{\star} = \frac{p}{p+g}, \qquad \max_D V = 2\,\mathrm{JS}(p, g) - 2\ln 2}
$$

**Key takeaway.** The factor $2$ here is the same factor as in Example 6.9: the divergence the
adversarial game minimizes is $2\,\mathrm{JS}$, whose ceiling is $2\ln 2$, so the plateau of
Problem L1.6 is where the generator's gradient dies.

In [21]:
p_gan = np.array([0.5, 0.3, 0.2])
g_gan = np.array([0.1, 0.2, 0.7])
D_star_gan = p_gan / (p_gan + g_gan)
V = float(np.sum(p_gan * np.log(D_star_gan)) + np.sum(g_gan * np.log(1 - D_star_gan)))
print(f"optimal discriminator D* = {D_star_gan}")
print(f"V(D*, G)                 = {V:.12f}")
print(f"2 JS - 2 ln 2            = {2 * js(p_gan, g_gan) - 2 * np.log(2.0):.12f}")
print(f"value at g = p           = {-2 * np.log(2.0):.6f}")
grid = np.linspace(1e-6, 1 - 1e-6, 20001)
best_pointwise = np.array([grid[np.argmax(p_gan[i] * np.log(grid) + g_gan[i] * np.log(1 - grid))]
                           for i in range(3)])
print(f"grid-search discriminator = {best_pointwise}   (matches D* to grid resolution)")
assert abs(V - (2 * js(p_gan, g_gan) - 2 * np.log(2.0))) < 1e-12
assert np.max(np.abs(best_pointwise - D_star_gan)) < 1e-4

optimal discriminator D* = [0.8333 0.6    0.2222]
V(D*, G)                 = -1.083578137976
2 JS - 2 ln 2            = -1.083578137976
value at g = p           = -1.386294
grid-search discriminator = [0.8333 0.6    0.2222]   (matches D* to grid resolution)


### Problem L2.5 — Monitoring distribution shift

**Statement.** A production feature is bucketed into four bins. Training frequencies are
$P = (0.4, 0.3, 0.2, 0.1)$; this week's traffic gives $Q = (0.25, 0.25, 0.25, 0.25)$. Compute
$D_{\mathrm{KL}}(Q \parallel P)$, the total variation, and the ceiling on a single-sample
detector's accuracy.

**Intuition.** A raw divergence number means nothing until it is converted into an accuracy or a
sample count.

**Solution.**

*Step 1 (divergence).*

$$
D_{\mathrm{KL}}(Q \parallel P) = 0.25\left(\ln\tfrac{0.25}{0.4} + \ln\tfrac{0.25}{0.3} + \ln\tfrac{0.25}{0.2} + \ln\tfrac{0.25}{0.1}\right)
$$

$$
= 0.25\left(-0.470004 - 0.182322 + 0.223144 + 0.916291\right) = 0.25 \times 0.487109 = 0.121777 \text{ nats}.
$$

*Step 2 (total variation).*
$\mathrm{TV} = \tfrac12(0.15 + 0.05 + 0.05 + 0.15) = 0.20$.

*Step 3 (detector ceiling).* The optimal single-sample test between the two has accuracy
$\tfrac12 + \tfrac12\mathrm{TV} = 0.60$, barely better than chance. Pinsker is consistent:
$\sqrt{0.121777/2} = 0.246756 \ge 0.20$.

*Step 4 (samples to detect).* By the exponent of Problem L3.3, driving the miss probability to
$\alpha = 0.05$ needs about $n \approx \ln(1/\alpha)/D_{\mathrm{KL}} = 2.995732/0.121777 \approx 25$
observations.

$$
\boxed{D_{\mathrm{KL}}(Q \parallel P) = 0.121777 \text{ nats}, \quad \mathrm{TV} = 0.200000, \quad \text{single-sample accuracy ceiling } 60\%}
$$

**Key takeaway.** Divergences become actionable only after translation — into detector accuracy via
$\mathrm{TV}$, or into samples-to-detect via the exponent.

In [22]:
P_train = np.array([0.4, 0.3, 0.2, 0.1])
Q_live = np.array([0.25, 0.25, 0.25, 0.25])
d_shift = kl(Q_live, P_train)
print(f"D(Q||P)                        = {d_shift:.6f} nats")
print(f"TV                             = {tv(Q_live, P_train):.6f}")
print(f"single-sample accuracy ceiling = {0.5 + 0.5 * tv(Q_live, P_train):.4f}")
print(f"Pinsker bound sqrt(KL/2)       = {np.sqrt(d_shift / 2):.6f}")
print(f"samples for alpha = 0.05       = {np.log(1 / 0.05) / d_shift:.2f}")
assert abs(d_shift - 0.121777) < 5e-7
assert tv(Q_live, P_train) <= np.sqrt(d_shift / 2)

D(Q||P)                        = 0.121777 nats
TV                             = 0.200000
single-sample accuracy ceiling = 0.6000
Pinsker bound sqrt(KL/2)       = 0.246756
samples for alpha = 0.05       = 24.60


### Problem L2.6 — Akaike's criterion estimates an expected relative entropy

**Statement.** Explain why $\mathrm{AIC} = -2\hat{\ell} + 2k$ ranks models by expected
$D_{\mathrm{KL}}$ from the truth, by deriving the in-sample optimism and showing it equals $k$.

**Intuition.** The fitted log-likelihood is optimistic because the parameter was chosen on the same
data; the correction is the expected size of that overfit, which is $k/2$ twice over.

**Solution.**

*Step 1 (the target).* For a fitted $q_{\hat\theta}$,
$D_{\mathrm{KL}}(p \parallel q_{\hat\theta}) = -H(p) - \mathbb{E}_{X \sim p}\left[\ln q_{\hat\theta}(X)\right]$,
and $H(p)$ does not depend on the model. Ranking by expected divergence is ranking by expected
log-likelihood on **fresh** data.

*Step 2 (notation).* Write $\ell_n(\theta) = \sum_{i=1}^{n}\ln q_\theta(x_i)$ for the in-sample
log-likelihood and $\ell(\theta) = n\,\mathbb{E}_{X\sim p}\left[\ln q_\theta(X)\right]$ for its
out-of-sample counterpart, $\theta_0$ for the maximizer of $\ell$, and $\hat\theta$ for the
maximizer of $\ell_n$. Under correct specification the curvature and the score covariance are both
the Fisher information $\mathcal{I}$.

*Step 3 (out-of-sample loss).* $\nabla\ell(\theta_0) = 0$ and $-\nabla^2\ell(\theta_0) = n\mathcal{I}$, so

$$
\ell(\theta_0) - \ell(\hat\theta) = \tfrac{n}{2}(\hat\theta - \theta_0)^{\top}\mathcal{I}(\hat\theta - \theta_0) + o_P(1).
$$

*Step 4 (in-sample gain).* Expanding $\ell_n$ about $\theta_0$ and using
$\hat\theta - \theta_0 \approx (n\mathcal{I})^{-1}\nabla \ell_n(\theta_0)$, the linear term equals
twice the quadratic one, leaving

$$
\ell_n(\hat\theta) - \ell_n(\theta_0) = \tfrac{n}{2}(\hat\theta - \theta_0)^{\top}\mathcal{I}(\hat\theta - \theta_0) + o_P(1).
$$

*Step 5 (add them).* $\mathbb{E}[\ell_n(\theta_0)] = \ell(\theta_0)$, so the optimism is the sum of
Steps 3 and 4:

$$
\mathbb{E}\left[\ell_n(\hat\theta) - \ell(\hat\theta)\right] = n\,\mathbb{E}\left[(\hat\theta-\theta_0)^{\top}\mathcal{I}(\hat\theta - \theta_0)\right] + o(1) \longrightarrow \mathbb{E}\left[\chi^2_k\right] = k,
$$

because $\sqrt{n}(\hat\theta - \theta_0) \to \mathcal{N}(0, \mathcal{I}^{-1})$. Half of the $k$ comes
from the in-sample gain and half from the out-of-sample loss.

*Step 6 (the criterion).* Correcting the bias and multiplying by $-2$ gives
$\mathrm{AIC} = -2(\hat\ell - k) = -2\hat\ell + 2k$.

$$
\boxed{\mathbb{E}\left[\hat\ell - \ell(\hat\theta)\right] = k + o(1), \qquad \mathrm{AIC} = -2\hat\ell + 2k}
$$

**Key takeaway.** AIC is not an arbitrary complexity penalty but a bias-corrected estimate of
predictive relative entropy, which is why it targets prediction rather than recovery of the true
model.

In [23]:
n_obs, reps = 200, 20000
print("Gaussian linear model, unit noise, truth independent of the regressors")
print("   k    observed optimism     s.e.    exact finite-n value    AIC penalty k")
for k in (1, 2, 4, 8):
    vals = np.empty(reps)
    for i in range(reps):
        X = rng.standard_normal((n_obs, k))
        y = rng.standard_normal(n_obs)
        beta = np.linalg.solve(X.T @ X, X.T @ y)
        in_sample = -0.5 * float(np.sum((y - X @ beta) ** 2))
        out_expected = -0.5 * n_obs * (1.0 + float(beta @ beta))
        vals[i] = in_sample - out_expected + 0.5 * (float(y @ y) - n_obs)   # zero-mean control variate
    exact = 0.5 * k * (1.0 + n_obs / (n_obs - k - 1))
    se = vals.std(ddof=1) / np.sqrt(reps)
    print(f"  {k:2d}      {vals.mean():8.4f}       {se:.4f}         {exact:8.4f}            {k}")
    assert abs(vals.mean() - exact) < 5 * se + 0.05

Gaussian linear model, unit noise, truth independent of the regressors
   k    observed optimism     s.e.    exact finite-n value    AIC penalty k


   1        0.9969       0.0100           1.0051            1


   2        1.9899       0.0142           2.0152            2


   4        4.0561       0.0203           4.0513            4


   8        8.1890       0.0290           8.1885            8


### Problem L2.7 — Landauer's bound for erasing a terabyte

**Statement.** A memory cell holding one unknown bit is reset to a known state at temperature
$T = 300$ K. Derive the minimum heat released, evaluate it, and compare the total for one terabyte
with a realistic DRAM write energy of $1$ pJ per bit.

**Intuition.** Erasure destroys one bit of entropy in the memory, and the second law makes the
environment absorb at least that much.

**Solution.**

*Step 1 (the divergence).* Before erasure the cell's state is uniform on $\{0,1\}$; after, it is a
point mass. Taking the uniform distribution as the equilibrium reference $\pi$, the reset state
$\delta$ has

$$
D_{\mathrm{KL}}(\delta \parallel \pi) = 1 \cdot \ln\frac{1}{1/2} = \ln 2 .
$$

*Step 2 (free energy).* By Problem L2.8 the nonequilibrium free energy of a state $p$ satisfies
$F(p) - F(\pi) = k_B T\, D_{\mathrm{KL}}(p \parallel \pi)$, so resetting the cell raises its free
energy by $k_B T \ln 2$, which must be supplied as work and released to the bath as heat.

*Step 3 (evaluate).* With $k_B = 1.380649 \times 10^{-23}$ J/K and $T = 300$ K,

$$
k_B T \ln 2 = 2.870979 \times 10^{-21} \text{ J per bit}.
$$

*Step 4 (one terabyte).* $8 \times 10^{12}$ bits cost $2.296783 \times 10^{-8}$ J, about $23$
nanojoules. A real DRAM at $1$ pJ per bit spends $8$ J for the same operation — a factor of
$3.48 \times 10^{8}$ above the thermodynamic floor.

$$
\boxed{Q_{\min} = k_B T \ln 2 = 2.870979 \times 10^{-21}\ \text{J per bit}; \quad \text{1 TB} \Rightarrow 2.296783 \times 10^{-8}\ \text{J}}
$$

**Key takeaway.** Landauer's bound is Theorem 4.1 with units attached: the nonnegativity of a
relative entropy becomes the statement that information erasure has an unavoidable energy price.

In [24]:
k_B = 1.380649e-23
T_K = 300.0
per_bit = k_B * T_K * np.log(2.0)
bits_per_TB = 8 * 10 ** 12
dram_per_bit = 1e-12
print(f"D_KL(delta || uniform)          = {kl(np.array([1.0, 0.0]), np.array([0.5, 0.5])):.6f} nats"
      f"  (= ln 2 = {np.log(2.0):.6f})")
print(f"Landauer minimum per bit at 300K = {per_bit:.6e} J")
print(f"one terabyte                     = {per_bit * bits_per_TB:.6e} J")
print(f"DRAM at 1 pJ per bit             = {dram_per_bit * bits_per_TB:.6e} J")
print(f"ratio, actual to thermodynamic   = {dram_per_bit / per_bit:.3e}")
assert abs(per_bit - 2.870979e-21) < 1e-27
assert abs(per_bit * bits_per_TB - 2.296783e-8) < 1e-14

D_KL(delta || uniform)          = 0.693147 nats  (= ln 2 = 0.693147)
Landauer minimum per bit at 300K = 2.870979e-21 J
one terabyte                     = 2.296783e-08 J
DRAM at 1 pJ per bit             = 8.000000e+00 J
ratio, actual to thermodynamic   = 3.483e+08


### Problem L2.8 — The Gibbs state is the free-energy minimizer

**Statement.** For a system with energy levels $E(x)$ at inverse temperature $\beta = 1/(k_B T)$,
show that the free energy $F(p) = \mathbb{E}_p[E] - T S(p)$ satisfies
$F(p) - F(\pi) = k_B T\, D_{\mathrm{KL}}(p \parallel \pi)$, where $\pi(x) \propto e^{-\beta E(x)}$,
and evaluate it for the three-level system $E = (0, 1, 2)$ in units of $k_B T$ with $p$ uniform.

**Intuition.** The Gibbs state is the unique minimizer of free energy, and relative entropy measures
exactly how far above the minimum any other state sits.

**Solution.**

*Step 1 (partition function).* $\pi(x) = e^{-\beta E(x)}/Z$ with $Z = \sum_x e^{-\beta E(x)}$, so
$E(x) = -k_B T\left(\ln \pi(x) + \ln Z\right)$.

*Step 2 (substitute).* With $S(p) = -k_B\sum_x p(x)\ln p(x)$,

$$
F(p) = \sum_x p(x) E(x) - T S(p) = -k_B T \sum_x p(x)\ln\pi(x) - k_B T \ln Z + k_B T \sum_x p(x)\ln p(x).
$$

*Step 3 (collect).* The first and third terms combine into $k_B T\, D_{\mathrm{KL}}(p \parallel \pi)$:

$$
F(p) = k_B T\, D_{\mathrm{KL}}(p \parallel \pi) - k_B T \ln Z .
$$

Setting $p = \pi$ gives $F(\pi) = -k_B T \ln Z$, so
$F(p) - F(\pi) = k_B T\, D_{\mathrm{KL}}(p \parallel \pi) \ge 0$ with equality only at $p = \pi$
(Theorem 4.1).

*Step 4 (numbers).* With $\beta = 1$ and $E = (0,1,2)$: $Z = 1 + e^{-1} + e^{-2} = 1.503215$,
$\pi = (0.665241, 0.244728, 0.090031)$, $F(\pi) = -\ln Z = -0.407606$. For $p$ uniform,
$F(p) = 1 - \ln 3 = -0.098612$, so the excess is $0.308994$, equal to
$D_{\mathrm{KL}}(p \parallel \pi)$.

$$
\boxed{F(p) - F(\pi) = k_B T\, D_{\mathrm{KL}}(p \parallel \pi) = 0.308994\, k_B T \text{ for the uniform state here}}
$$

**Key takeaway.** Relaxation to equilibrium is descent in relative entropy, and the second law for
this system is exactly Gibbs' inequality.

In [25]:
E_levels = np.array([0.0, 1.0, 2.0])       # in units of k_B T, so beta = 1
Z = float(np.sum(np.exp(-E_levels)))
pi_gibbs = np.exp(-E_levels) / Z
p_unif = np.full(3, 1.0 / 3.0)
free_energy = lambda p: float(p @ E_levels) + float(np.sum(p * np.log(p)))
print(f"Z = {Z:.6f}   pi = {pi_gibbs}")
print(f"F(pi)     = {free_energy(pi_gibbs):.6f}   -ln Z = {-np.log(Z):.6f}")
print(f"F(uniform)= {free_energy(p_unif):.6f}   1 - ln 3 = {1 - np.log(3):.6f}")
print(f"F(p) - F(pi) = {free_energy(p_unif) - free_energy(pi_gibbs):.6f}"
      f"   D_KL(p||pi) = {kl(p_unif, pi_gibbs):.6f}")
worst_fe = 0.0
for _ in range(500):
    pr = rng.dirichlet(np.ones(3))
    worst_fe = max(worst_fe, abs((free_energy(pr) - free_energy(pi_gibbs)) - kl(pr, pi_gibbs)))
print(f"worst |F(p) - F(pi) - D_KL| over 500 random states = {worst_fe:.3e} = {worst_fe / EPS:.1f} * eps")
assert abs(free_energy(p_unif) - free_energy(pi_gibbs) - 0.308994) < 5e-7
assert worst_fe < 50 * EPS

Z = 1.503215   pi = [0.6652 0.2447 0.09  ]
F(pi)     = -0.407606   -ln Z = -0.407606
F(uniform)= -0.098612   1 - ln 3 = -0.098612
F(p) - F(pi) = 0.308994   D_KL(p||pi) = 0.308994
worst |F(p) - F(pi) - D_KL| over 500 random states = 4.441e-16 = 2.0 * eps


### Problem L2.9 — Two Maxwell-Boltzmann gases at different temperatures

**Statement.** The velocity of a molecule of mass $m$ in an ideal gas at temperature $T$ is
$\mathcal{N}(0, k_B T/m)$ in each of three Cartesian components. Compute
$D_{\mathrm{KL}}$ between the velocity distributions at $T_1 = 300$ K and $T_2 = 350$ K, and
estimate how many molecules must be sampled to tell the two gases apart.

**Intuition.** Three independent coordinates make the divergence additive, and each coordinate is
the equal-mean Gaussian case of Theorem 4.6.

**Solution.**

*Step 1 (one component).* Theorem 4.6 with $\mu_1 = \mu_2 = 0$ and
$\sigma_i^2 = k_B T_i / m$ gives

$$
D_{\mathrm{KL}} = \tfrac{1}{2}\ln\frac{T_2}{T_1} + \frac{T_1}{2T_2} - \frac{1}{2}.
$$

*Step 2 (three components).* By Theorem 4.5 the three coordinates add:

$$
D_{\mathrm{KL}}\bigl(f_{T_1} \parallel f_{T_2}\bigr) = \frac{3}{2}\left[\ln\frac{T_2}{T_1} + \frac{T_1}{T_2} - 1\right],
$$

which is independent of the mass — the ratio $T_1/T_2$ is the only thing that matters.

*Step 3 (numbers).* $\ln(350/300) = 0.154151$ and $300/350 = 0.857143$, so the bracket is
$0.011294$ and the divergence is $0.016940$ nats per molecule. In the other direction it is
$0.018774$ nats.

*Step 4 (sample size).* By the exponent of Problem L3.3, reaching a miss probability of $0.05$
needs about $n \approx \ln(1/0.05)/0.016940 \approx 177$ molecules.

$$
\boxed{D_{\mathrm{KL}} = \tfrac{3}{2}\left[\ln\tfrac{T_2}{T_1} + \tfrac{T_1}{T_2} - 1\right] = 0.016940 \text{ nats per molecule}}
$$

**Key takeaway.** A $17$ percent temperature difference is worth only $0.017$ nats per molecule, so
thermometry by velocity sampling needs hundreds of molecules — the operational content of a small
relative entropy.

In [26]:
T1, T2 = 300.0, 350.0
per_component = 0.5 * np.log(T2 / T1) + T1 / (2 * T2) - 0.5
three_d = 1.5 * (np.log(T2 / T1) + T1 / T2 - 1.0)
print(f"per component            = {per_component:.6f} nats")
print(f"three components         = {3 * per_component:.6f} nats   closed form = {three_d:.6f}")
print(f"reverse direction (3D)   = {1.5 * (np.log(T1 / T2) + T2 / T1 - 1.0):.6f} nats")
sigma1, sigma2 = np.sqrt(T1), np.sqrt(T2)      # k_B / m absorbed: only the ratio matters
print(f"check against Theorem 4.6: {3 * gauss_kl(0.0, sigma1, 0.0, sigma2):.6f} nats")
print(f"molecules for alpha = 0.05: {np.log(1 / 0.05) / three_d:.1f}")
assert abs(3 * per_component - three_d) < 1e-12
assert abs(3 * gauss_kl(0.0, sigma1, 0.0, sigma2) - three_d) < 1e-12
assert abs(three_d - 0.016940) < 5e-7

per component            = 0.005647 nats
three components         = 0.016940 nats   closed form = 0.016940
reverse direction (3D)   = 0.018774 nats
check against Theorem 4.6: 0.016940 nats
molecules for alpha = 0.05: 176.8


## L3 — Challenge Proofs

### Problem L3.1 — The Donsker-Varadhan variational formula

**Statement.** Prove

$$
D_{\mathrm{KL}}(P \parallel Q) = \sup_{T}\left\{\mathbb{E}_P\left[T(X)\right] - \ln\mathbb{E}_Q\left[e^{T(X)}\right]\right\}
$$

over all $T$ with $\mathbb{E}_Q\left[e^{T}\right] \lt \infty$, and identify the optimizer.

**Intuition.** Every candidate $T$ tilts $Q$ into a new distribution, and the gap in the formula is
exactly the divergence from $P$ to that tilted distribution.

**Solution.**

*Step 1 (tilt).* For admissible $T$ define
$g_T(x) = \dfrac{q(x)e^{T(x)}}{\mathbb{E}_Q\left[e^{T}\right]}$, a probability distribution.

*Step 2 (the gap is a divergence).* Expanding $D_{\mathrm{KL}}(P \parallel G_T)$,

$$
D_{\mathrm{KL}}(P \parallel G_T) = \mathbb{E}_P\left[\ln\tfrac{p(X)}{q(X)}\right] - \mathbb{E}_P\left[T(X)\right] + \ln\mathbb{E}_Q\left[e^{T}\right],
$$

that is
$D_{\mathrm{KL}}(P \parallel Q) - \left(\mathbb{E}_P[T] - \ln\mathbb{E}_Q[e^{T}]\right) = D_{\mathrm{KL}}(P \parallel G_T)$.

*Step 3 (bound).* The right side is nonnegative by Theorem 4.1, so every $T$ gives a lower bound and
the supremum is at most $D_{\mathrm{KL}}(P \parallel Q)$.

*Step 4 (tightness).* Equality needs $G_T = P$, that is
$T^{\star}(x) = \ln\frac{p(x)}{q(x)} + c$ for any constant $c$, which cancels between the two terms.

$$
\boxed{D_{\mathrm{KL}}(P \parallel Q) = \sup_T\left\{\mathbb{E}_P[T] - \ln\mathbb{E}_Q\left[e^{T}\right]\right\}, \qquad T^{\star} = \ln\frac{p}{q} + c}
$$

**Key takeaway.** An intractable density-ratio integral becomes a trainable optimization. Two
caveats fall out of the proof: any finite-capacity $T_\theta$ gives a **lower** bound, and the
plug-in $\ln$ of a sample mean is biased downward by Jensen.

In [27]:
Pv = np.array([0.5, 0.3, 0.2])
Qv = np.array([0.1, 0.2, 0.7])
T_star = np.log(Pv / Qv)
dv = lambda T: float(Pv @ T) - np.log(float(Qv @ np.exp(T)))
print(f"DV objective at T* = ln(p/q) : {dv(T_star):.12f}")
print(f"D_KL(P||Q)                   : {kl(Pv, Qv):.12f}")
print(f"shift invariance, T* + 3      : {dv(T_star + 3.0):.12f}")
best = -np.inf
for _ in range(20000):
    T = T_star + rng.standard_normal(3) * 0.5
    best = max(best, dv(T))
print(f"best over 20000 random perturbations of T*: {best:.12f}  (never exceeds D_KL)")
assert abs(dv(T_star) - kl(Pv, Qv)) < 1e-12
assert abs(dv(T_star + 3.0) - kl(Pv, Qv)) < 1e-12
assert best <= kl(Pv, Qv) + 1e-12

DV objective at T* = ln(p/q) : 0.675805894950
D_KL(P||Q)                   : 0.675805894950
shift invariance, T* + 3      : 0.675805894950
best over 20000 random perturbations of T*: 0.675804001732  (never exceeds D_KL)


### Problem L3.2 — The Fenchel variational bound for any $f$-divergence

**Statement.** With $f^{\ast}(u) = \sup_t \{ut - f(t)\}$, prove

$$
D_f(P \parallel Q) \;\ge\; \sup_T \left\{\mathbb{E}_P\left[T(X)\right] - \mathbb{E}_Q\left[f^{\ast}(T(X))\right]\right\},
$$

with equality at $T^{\star} = f'(p/q)$, and specialize to the adversarial objective.

**Intuition.** Replace $f$ by its biconjugate and pull the pointwise supremum outside the sum.

**Solution.**

*Step 1 (biconjugation).* $f$ is convex and lower semicontinuous, so $f = f^{\ast\ast}$, that is
$f(t) = \sup_u\{ut - f^{\ast}(u)\}$.

*Step 2 (insert).* With $t(x) = p(x)/q(x)$ and any fixed $T$,

$$
D_f(P \parallel Q) = \sum_x q(x)\sup_u\left\{u\tfrac{p(x)}{q(x)} - f^{\ast}(u)\right\}
\;\ge\; \sum_x q(x)\left\{T(x)\tfrac{p(x)}{q(x)} - f^{\ast}(T(x))\right\},
$$

which simplifies to $\mathbb{E}_P[T] - \mathbb{E}_Q\left[f^{\ast}(T)\right]$.

*Step 3 (tightness).* The pointwise supremum is attained at $u = f'(t)$, so
$T^{\star}(x) = f'\!\left(p(x)/q(x)\right)$ makes every inequality an equality.

*Step 4 (adversarial specialization).* Take $f(t) = t\ln t - (t+1)\ln(t+1) + 2\ln 2$, whose
divergence is $2\,\mathrm{JS}$ — the **unhalved** Jensen-Shannon generator of Example 6.9, shifted
by the affine term $2\ln 2 \cdot 1$ so that $f(1) = 0$. Parameterizing $T = \ln D$ with
$D \in (0,1)$ and computing $f^{\ast}$ reproduces

$$
\sup_D\left\{\mathbb{E}_P\left[\ln D(X)\right] + \mathbb{E}_Q\left[\ln(1 - D(X))\right]\right\} + 2\ln 2 = 2\,\mathrm{JS}(P, Q),
$$

which is Problem L2.4 seen as one instance of a family.

$$
\boxed{D_f(P \parallel Q) = \sup_T\left\{\mathbb{E}_P[T] - \mathbb{E}_Q\left[f^{\ast}(T)\right]\right\}, \qquad T^{\star} = f'\!\left(\tfrac{p}{q}\right)}
$$

**Key takeaway.** Every $f$-divergence has a discriminator-shaped dual, so choosing $f$ chooses the
loss, the output activation and the failure mode of the adversarial game. The factor of two is not
optional: the generator above produces $2\,\mathrm{JS}$, not $\mathrm{JS}$.

In [28]:
f_kl = lambda t: t * np.log(t)
fstar_kl = lambda u: np.exp(u - 1.0)                # conjugate of t ln t
T_opt = np.log(Pv / Qv) + 1.0                       # f'(t) = ln t + 1
lhs = kl(Pv, Qv)
rhs = float(Pv @ T_opt) - float(Qv @ fstar_kl(T_opt))
print(f"KL case:   D_f = {lhs:.12f}   Fenchel value at T* = f'(p/q) = {rhs:.12f}")
best = -np.inf
for _ in range(20000):
    T = T_opt + rng.standard_normal(3) * 0.4
    best = max(best, float(Pv @ T) - float(Qv @ fstar_kl(T)))
print(f"best over 20000 perturbations of T*: {best:.12f}  (never exceeds D_f)")

f_gan = lambda t: t * np.log(t) - (t + 1) * np.log(t + 1) + 2 * np.log(2.0)
d_gan = float(np.sum(Qv * f_gan(Pv / Qv)))
print(f"\nGAN generator: D_f = {d_gan:.12f}   2 JS = {2 * js(Pv, Qv):.12f}"
      f"   JS = {js(Pv, Qv):.12f}")
D_opt = Pv / (Pv + Qv)
V_opt = float(np.sum(Pv * np.log(D_opt)) + np.sum(Qv * np.log(1 - D_opt)))
print(f"sup_D V + 2 ln 2   = {V_opt + 2 * np.log(2.0):.12f}")
assert abs(rhs - lhs) < 1e-12 and best <= lhs + 1e-12
assert abs(d_gan - 2 * js(Pv, Qv)) < 1e-12
assert abs(V_opt + 2 * np.log(2.0) - 2 * js(Pv, Qv)) < 1e-12

KL case:   D_f = 0.675805894950   Fenchel value at T* = f'(p/q) = 0.675805894950
best over 20000 perturbations of T*: 0.675735748693  (never exceeds D_f)

GAN generator: D_f = 0.302716223144   2 JS = 0.302716223144   JS = 0.151358111572
sup_D V + 2 ln 2   = 0.302716223144


### Problem L3.3 — Chernoff-Stein: relative entropy as a testing exponent

**Statement.** Test $H_0 : X^n \sim P^{\otimes n}$ against $H_1 : X^n \sim Q^{\otimes n}$ with the
type-I error constrained by $\alpha_n \le \epsilon$. Show that the optimal type-II error satisfies
$\tfrac{1}{n}\ln\beta_n \to -D_{\mathrm{KL}}(P \parallel Q)$.

**Intuition.** The log-likelihood ratio is a sum of i.i.d. terms with mean $D_{\mathrm{KL}}$, so the
law of large numbers pins the exponent from both sides.

**Solution.**

*Step 1.* Let $L_n = \sum_{i=1}^{n}\ln\frac{p(X_i)}{q(X_i)}$. Under $P$ the summands are i.i.d. with
mean $D_{\mathrm{KL}}(P \parallel Q)$, so $\tfrac1n L_n \to D_{\mathrm{KL}}$ in probability.

*Step 2 (achievability).* Accept $H_0$ on
$A_n = \{x^n : \tfrac1n L_n \gt D_{\mathrm{KL}} - \delta\}$. Step 1 gives $P^{\otimes n}(A_n) \to 1$,
so the type-I error is eventually below $\epsilon$. On $A_n$ the likelihood ratio exceeds
$e^{n(D_{\mathrm{KL}} - \delta)}$, so

$$
1 \;\ge\; P^{\otimes n}(A_n) = \sum_{x^n \in A_n} q^n(x^n)\frac{p^n(x^n)}{q^n(x^n)} \;\ge\; e^{n(D_{\mathrm{KL}} - \delta)} Q^{\otimes n}(A_n),
$$

hence $\beta_n \le e^{-n(D_{\mathrm{KL}} - \delta)}$.

*Step 3 (converse).* For any acceptance region $B_n$ with $P^{\otimes n}(B_n) \ge 1 - \epsilon$,
split it by whether $\tfrac1n L_n \le D_{\mathrm{KL}} + \delta$. The other part has vanishing
$P$-probability, so $P^{\otimes n}\bigl(B_n \cap \{\tfrac1n L_n \le D_{\mathrm{KL}} + \delta\}\bigr) \ge 1 - \epsilon - o(1)$,
and on that event $q^n \ge p^n e^{-n(D_{\mathrm{KL}} + \delta)}$, giving
$\beta_n \ge (1 - \epsilon - o(1))e^{-n(D_{\mathrm{KL}} + \delta)}$.

*Step 4.* Take $\tfrac1n\ln$ of both bounds and let $\delta \to 0$.

$$
\boxed{\lim_{n\to\infty}\frac{1}{n}\ln\beta_n = -D_{\mathrm{KL}}(P \parallel Q)}
$$

**Key takeaway.** Relative entropy is not merely *a* measure of dissimilarity; it is *the*
exponential rate at which evidence accumulates, which is why it and not $\mathrm{TV}$ indexes
sample complexity.

In [29]:
Pt, Qt = np.array([0.7, 0.3]), np.array([0.5, 0.5])
D_pq = kl(Pt, Qt)
c_slope = np.log(0.7 / 0.5) - np.log(0.3 / 0.5)


def log_binom_tail(n, k, prob):
    """log P(Bin(n, prob) >= k), by log-sum-exp over the summands."""
    j = np.arange(k, n + 1)
    log_terms = (special.gammaln(n + 1) - special.gammaln(j + 1) - special.gammaln(n - j + 1)
                 + j * np.log(prob) + (n - j) * np.log1p(-prob))
    return float(special.logsumexp(log_terms))


print(f"testing Ber(0.7) against Ber(0.5);  D(P||Q) = {D_pq:.6f} nats")
print("acceptance region A_n = { L_n / n > D - delta }, with n chosen as 20 / delta^2")
print("   delta        n       threshold p*     type-I alpha_n    -ln(beta_n)/n     D - delta")
for delta in (0.05, 0.02, 0.01, 0.005, 0.002):
    n_t = int(20 / delta ** 2)
    p_star = (D_pq - delta - np.log(0.3 / 0.5)) / c_slope
    k_t = int(np.ceil(p_star * n_t))
    alpha_n = float(stats.binom.cdf(k_t - 1, n_t, 0.7))
    expo = -log_binom_tail(n_t, k_t, 0.5) / n_t
    print(f"  {delta:6.3f}  {n_t:9d}      {p_star:.6f}       {alpha_n:.3e}       "
          f"{expo:.6f}       {D_pq - delta:.6f}")
    assert alpha_n < 0.05
    assert D_pq - delta <= expo <= D_pq + delta
print(f"\nthe measured exponent is sandwiched in [D - delta, D + delta] at every delta,")
print(f"and rises towards D = {D_pq:.6f} as delta shrinks, which is the statement of the theorem")

testing Ber(0.7) against Ber(0.5);  D(P||Q) = 0.082283 nats
acceptance region A_n = { L_n / n > D - delta }, with n chosen as 20 / delta^2
   delta        n       threshold p*     type-I alpha_n    -ln(beta_n)/n     D - delta
   0.050       7999      0.640989       5.153e-30       0.040835       0.032283
   0.020      50000      0.676396       1.338e-30       0.063692       0.062283
   0.010     200000      0.688198       8.683e-31       0.072642       0.072283


   0.005     800000      0.694099       6.934e-31       0.077374       0.077283


   0.002    5000000      0.697640       5.906e-31       0.080298       0.080283

the measured exponent is sandwiched in [D - delta, D + delta] at every delta,
and rises towards D = 0.082283 as delta shrinks, which is the statement of the theorem


### Problem L3.4 — Relative entropy as a Bregman divergence, and the Fisher metric

**Statement.** (a) For an exponential family $p_\theta(x) = h(x)e^{\theta^{\top}\phi(x) - A(\theta)}$,
show that $D_{\mathrm{KL}}(p_{\theta_1} \parallel p_{\theta_2})$ is the Bregman divergence of the
log-partition function $A$. (b) Deduce
$D_{\mathrm{KL}}(p_\theta \parallel p_{\theta+\delta}) = \tfrac12\delta^{\top}\mathcal{I}(\theta)\delta + O(\lVert \delta \rVert^3)$.

**Intuition.** Inside an exponential family the log-ratio is affine in the sufficient statistic, so
everything reduces to the geometry of one convex function.

**Solution.**

*(a) Step 1.*
$\ln\frac{p_{\theta_1}(x)}{p_{\theta_2}(x)} = (\theta_1-\theta_2)^{\top}\phi(x) - A(\theta_1) + A(\theta_2)$.

*(a) Step 2.* Take the expectation under $p_{\theta_1}$ and use
$\mathbb{E}_{\theta_1}[\phi(X)] = \nabla A(\theta_1)$:

$$
D_{\mathrm{KL}}(p_{\theta_1} \parallel p_{\theta_2}) = A(\theta_2) - A(\theta_1) - \nabla A(\theta_1)^{\top}(\theta_2 - \theta_1) = B_A(\theta_2, \theta_1),
$$

the Bregman divergence generated by $A$ — with the arguments reversed, which is the source of the
forward/reverse bookkeeping in information geometry.

*(b) Step 3.* Bregman divergences vanish to first order, so expanding $A$ at $\theta$ with
$\theta_2 = \theta + \delta$, $\theta_1 = \theta$:
$B_A(\theta+\delta, \theta) = \tfrac12\delta^{\top}\nabla^2 A(\theta)\delta + O(\lVert \delta \rVert^3)$.

*(b) Step 4.* For exponential families
$\nabla^2 A(\theta) = \operatorname{Cov}_\theta[\phi(X)] = \mathcal{I}(\theta)$.

$$
\boxed{D_{\mathrm{KL}}(p_{\theta_1} \parallel p_{\theta_2}) = B_A(\theta_2, \theta_1), \qquad D_{\mathrm{KL}}(p_\theta \parallel p_{\theta+\delta}) \approx \tfrac12\delta^{\top}\mathcal{I}(\theta)\delta}
$$

**Key takeaway.** Locally relative entropy is a quadratic form with the Fisher information as its
matrix — the justification for natural gradient descent and for KL trust regions. Section 7.3
measures the exponent of that expansion as $1.9853$ against the predicted $2$.

In [30]:
# Bernoulli as an exponential family: theta = logit(p), A(theta) = ln(1 + e^theta)
A = lambda th: np.log1p(np.exp(th))
dA = lambda th: 1.0 / (1.0 + np.exp(-th))
d2A = lambda th: dA(th) * (1 - dA(th))
th1, th2 = np.log(0.3 / 0.7), np.log(0.55 / 0.45)
breg = A(th2) - A(th1) - dA(th1) * (th2 - th1)
print(f"Bregman B_A(theta2, theta1) = {breg:.12f}")
print(f"D_KL(Ber(0.3) || Ber(0.55)) = {kl(np.array([0.3, 0.7]), np.array([0.55, 0.45])):.12f}")
print("\nlocal quadratic expansion at theta = logit(0.3):")
for eps_th in (1e-1, 1e-2, 1e-3, 1e-4):
    exact = A(th1 + eps_th) - A(th1) - dA(th1) * eps_th
    quad = 0.5 * d2A(th1) * eps_th ** 2
    print(f"  delta = {eps_th:.0e}   exact = {exact:.12e}   0.5 I delta^2 = {quad:.12e}"
          f"   ratio = {exact / quad:.8f}")
assert abs(breg - kl(np.array([0.3, 0.7]), np.array([0.55, 0.45]))) < 1e-12

Bregman B_A(theta2, theta1) = 0.127442185524
D_KL(Ber(0.3) || Ber(0.55)) = 0.127442185524

local quadratic expansion at theta = logit(0.3):
  delta = 1e-01   exact = 1.063761864891e-03   0.5 I delta^2 = 1.050000000000e-03   ratio = 1.01310654
  delta = 1e-02   exact = 1.051397714358e-05   0.5 I delta^2 = 1.050000000000e-05   ratio = 1.00133116
  delta = 1e-03   exact = 1.050139977423e-07   0.5 I delta^2 = 1.050000000000e-07   ratio = 1.00013331
  delta = 1e-04   exact = 1.050013994660e-09   0.5 I delta^2 = 1.050000000000e-09   ratio = 1.00001333


### Problem L3.5 — The Bretagnolle-Huber inequality

**Statement.** Prove

$$
\mathrm{TV}(P, Q) \;\le\; \sqrt{1 - e^{-D_{\mathrm{KL}}(P \parallel Q)}},
$$

and explain when it beats Pinsker.

**Intuition.** Total variation is bounded by the Bhattacharyya coefficient, and Problem L1.9 bounds
that coefficient below by $e^{-D_{\mathrm{KL}}/2}$.

**Solution.**

*Step 1.* From Problem L1.8, $2\,\mathrm{TV} \le H\sqrt{4 - H^2}$ with
$H^2 = 2 - 2\,\mathrm{BC}$. Substituting,

$$
H^2(4 - H^2) = (2 - 2\mathrm{BC})(2 + 2\mathrm{BC}) = 4\left(1 - \mathrm{BC}^2\right),
$$

so $\mathrm{TV}^2 \le 1 - \mathrm{BC}^2$.

*Step 2.* Problem L1.9 Step 2 gives $\mathrm{BC} \ge e^{-D_{\mathrm{KL}}/2}$, hence
$\mathrm{BC}^2 \ge e^{-D_{\mathrm{KL}}}$.

*Step 3.* Combining, $\mathrm{TV}^2 \le 1 - e^{-D_{\mathrm{KL}}}$.

*Step 4 (comparison).* Pinsker gives $\sqrt{D_{\mathrm{KL}}/2}$, which exceeds $1$ — and is
therefore vacuous — once $D_{\mathrm{KL}} \gt 2$, while Bretagnolle-Huber never exceeds $1$. The
crossover is at the solution of $D/2 = 1 - e^{-D}$, namely $D \approx 1.5936$: below it Pinsker is
tighter, above it Bretagnolle-Huber is.

$$
\boxed{\mathrm{TV}(P,Q) \le \sqrt{1 - e^{-D_{\mathrm{KL}}(P \parallel Q)}} \;\le\; 1}
$$

**Key takeaway.** Pinsker is the right tool for the near-identical regime and Bretagnolle-Huber for
the well-separated one; minimax lower bounds use the second because they need a bound that stays
below $1$ when the divergence is large.

In [31]:
from scipy.optimize import brentq

worst_bh = np.inf
n_pinsker_better = n_bh_better = 0
for _ in range(4000):
    pa = rng.dirichlet(np.ones(4))
    qa = rng.dirichlet(np.ones(4))
    d = kl(pa, qa)
    bh = np.sqrt(1 - np.exp(-d))
    pin = np.sqrt(d / 2)
    worst_bh = min(worst_bh, bh - tv(pa, qa))
    if pin < bh:
        n_pinsker_better += 1
    else:
        n_bh_better += 1
print(f"smallest  BH bound - TV  over 4000 pairs = {worst_bh:.3e}  (never negative)")
print(f"Pinsker tighter in {n_pinsker_better} cases, Bretagnolle-Huber tighter in {n_bh_better}")
cross = brentq(lambda d: d / 2 - (1 - np.exp(-d)), 0.5, 5.0)
print(f"crossover D_KL where the two bounds agree = {cross:.6f}")
print(f"at D_KL = 3: Pinsker = {np.sqrt(3 / 2):.6f} (vacuous), "
      f"BH = {np.sqrt(1 - np.exp(-3.0)):.6f}")
assert worst_bh > -1e-12
assert abs(cross - 1.5936) < 1e-3

smallest  BH bound - TV  over 4000 pairs = 6.257e-03  (never negative)
Pinsker tighter in 3575 cases, Bretagnolle-Huber tighter in 425
crossover D_KL where the two bounds agree = 1.593624
at D_KL = 3: Pinsker = 1.224745 (vacuous), BH = 0.974789


### Problem L3.6 — Renyi divergence is nondecreasing in its order

**Statement.** For $\alpha \gt 0$, $\alpha \neq 1$, define

$$
D_\alpha(P \parallel Q) = \frac{1}{\alpha - 1}\ln \sum_x p(x)^{\alpha} q(x)^{1-\alpha}.
$$

Show that $\alpha \mapsto D_\alpha$ is nondecreasing, that $D_\alpha \to D_{\mathrm{KL}}$ as
$\alpha \to 1$, and identify $D_{1/2}$ and $D_2$.

**Intuition.** Write the definition in terms of the cumulant generating function of the
log-likelihood ratio; the claim becomes the statement that a convex function through the origin has
nondecreasing slope from the origin.

**Solution.**

*Step 1 (rewrite).* Let $Z = p(X)/q(X)$ with $X \sim P$, and put $s = \alpha - 1$. Then

$$
\sum_x p^{\alpha}q^{1-\alpha} = \mathbb{E}_P\left[Z^{\alpha-1}\right] = \mathbb{E}_P\left[e^{s \ln Z}\right] = e^{\Lambda(s)},
$$

where $\Lambda(s) = \ln\mathbb{E}_P\left[e^{s\ln Z}\right]$ is the cumulant generating function of
$\ln Z$ under $P$. So $D_\alpha = \Lambda(s)/s$.

*Step 2 (convexity).* $\Lambda$ is convex by Holder, and $\Lambda(0) = 0$.

*Step 3 (slope from the origin).* For a convex $\Lambda$ with $\Lambda(0) = 0$, the difference
quotient $s \mapsto \Lambda(s)/s = \frac{\Lambda(s) - \Lambda(0)}{s - 0}$ is nondecreasing. Hence
$D_\alpha$ is nondecreasing in $\alpha$.

*Step 4 (the limit).* $\Lambda$ is differentiable at $0$ with
$\Lambda'(0) = \mathbb{E}_P[\ln Z] = D_{\mathrm{KL}}(P \parallel Q)$, so
$D_\alpha = \Lambda(s)/s \to \Lambda'(0) = D_{\mathrm{KL}}$ as $s \to 0$.

*Step 5 (two members).* At $\alpha = \tfrac12$,
$D_{1/2} = -2\ln\sum_x\sqrt{p q} = -2\ln \mathrm{BC}$, the Bhattacharyya divergence. At
$\alpha = 2$, $D_2 = \ln\sum_x \frac{p^2}{q} = \ln\left(1 + \chi^2(P \parallel Q)\right)$, which is
the middle term of Problem L1.5.

$$
\boxed{\alpha \mapsto D_\alpha(P \parallel Q) \text{ is nondecreasing}; \quad D_1 = D_{\mathrm{KL}}, \quad D_{1/2} = -2\ln\mathrm{BC}, \quad D_2 = \ln(1 + \chi^2)}
$$

**Key takeaway.** The Renyi family interpolates the inequalities of Problems L1.5 and L1.9 into one
monotone curve: $\alpha = \tfrac12$ recovers Hellinger, $\alpha = 1$ relative entropy, and
$\alpha = 2$ the $\chi^2$ bound, and monotonicity explains why they always appear in that order.

In [32]:
def renyi(p, q, alpha):
    p = np.asarray(p, float)
    q = np.asarray(q, float)
    if abs(alpha - 1.0) < 1e-12:
        return kl(p, q)
    return float(np.log(np.sum(p ** alpha * q ** (1 - alpha))) / (alpha - 1.0))


alphas = np.array([0.1, 0.25, 0.5, 0.75, 0.9, 1.0, 1.25, 1.5, 2.0, 3.0, 5.0])
vals = np.array([renyi(Pv, Qv, a) for a in alphas])
print("  alpha    D_alpha(P||Q)")
for a, v in zip(alphas, vals):
    print(f"  {a:5.2f}     {v:.6f}")
print(f"\nmonotone nondecreasing: {bool(np.all(np.diff(vals) >= -1e-12))}")
BC = float(np.sum(np.sqrt(Pv * Qv)))
print(f"D_1/2 = {renyi(Pv, Qv, 0.5):.8f}   -2 ln BC = {-2 * np.log(BC):.8f}")
print(f"D_1   = {renyi(Pv, Qv, 1.0):.8f}   D_KL     = {kl(Pv, Qv):.8f}")
print(f"D_2   = {renyi(Pv, Qv, 2.0):.8f}   ln(1+chi2) = {np.log1p(chi2(Pv, Qv)):.8f}")
print(f"limit check, alpha = 1 +/- 1e-6: {renyi(Pv, Qv, 1 - 1e-6):.8f}, {renyi(Pv, Qv, 1 + 1e-6):.8f}")
assert np.all(np.diff(vals) >= -1e-12)
assert abs(renyi(Pv, Qv, 0.5) + 2 * np.log(BC)) < 1e-12
assert abs(renyi(Pv, Qv, 2.0) - np.log1p(chi2(Pv, Qv))) < 1e-12
assert abs(renyi(Pv, Qv, 1 + 1e-6) - kl(Pv, Qv)) < 1e-5

  alpha    D_alpha(P||Q)
   0.10     0.064839
   0.25     0.166438
   0.50     0.342237
   0.75     0.515849
   0.90     0.614057
   1.00     0.675806
   1.25     0.814705
   1.50     0.930437
   2.00     1.100990
   3.00     1.289780
   5.00     1.437364

monotone nondecreasing: True
D_1/2 = 0.34223746   -2 ln BC = 0.34223746
D_1   = 0.67580589   D_KL     = 0.67580589
D_2   = 1.10099041   ln(1+chi2) = 1.10099041
limit check, alpha = 1 +/- 1e-6: 0.67580529, 0.67580650
